In [ ]:
#!/usr/bin/env python3
"""
Complete Elevation Data Pipeline with Coordinate Transformation
Searches for tiles with early LiDAR interpolation check and proper CRS handling
Priority: 1) LiDAR, 2) Sub-meter DEM, 3) 1-meter DEM, 4) 10-meter DEM, 5) OpenTopography
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
import xml.etree.ElementTree as ET
import zipfile
import tempfile
import re
import os
import requests
import time
import math
import json
import pickle
import gzip
import shutil
from datetime import datetime
import hashlib
import pandas as pd
from typing import List, Tuple, Dict, Optional, Set
from dataclasses import dataclass, field
from pathlib import Path
import warnings



# Import optional libraries with fallback
try:
    import laspy
    LASPY_AVAILABLE = True
except ImportError:
    LASPY_AVAILABLE = False
    print("Warning: laspy not installed. Install with: pip install laspy")

try:
    import rasterio
    from rasterio.warp import calculate_default_transform, reproject, Resampling
    RASTERIO_AVAILABLE = True
except ImportError:
    RASTERIO_AVAILABLE = False
    print("Warning: rasterio not installed. Install with: pip install rasterio")

try:
    from pyproj import Transformer, CRS
    PYPROJ_AVAILABLE = True
except ImportError:
    PYPROJ_AVAILABLE = False
    print("Warning: pyproj not installed. Install with: pip install pyproj")
    print("         This is REQUIRED for LiDAR coordinate transformation!")

try:
    import stateplane
    STATEPLANE_AVAILABLE = True
except ImportError:
    STATEPLANE_AVAILABLE = False
    print("Warning: stateplane not installed. Install with: pip install stateplane")
    print("         Falling back to coordinate-based detection.")

from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter1d
try:
    import simplekml
    SIMPLEKML_AVAILABLE = True
except ImportError:
    SIMPLEKML_AVAILABLE = False

from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ==================== CONFIGURATION ====================

# Input KMZ file path
INPUT_KMZ = r"C:\Users\peckb\Documents\TRIMET\CVG-Express.kmz"

# NEW: Bridge/Tunnel KMZ file path (set to None if not using)
BRIDGE_TUNNEL_KMZ = r"C:\Users\peckb\Documents\TRIMET\Bridges-Viaducts.kmz"  # Set this to your bridge/tunnel KMZ path, e.g., r"C:\path\to\bridges.kmz"

# Path handling configuration
PATH_HANDLING = 'all'  # Options: 'connect', 'longest', 'first', 'last', 'all'
# 'connect' - Connect all paths end-to-end
# 'longest' - Use only the longest path
# 'first' - Use only the first path
# 'last' - Use only the last path
# 'all' - Process all paths but warn about gaps

# Interpolation spacing
INTERPOLATION_SPACING_M = 10.0  # 10 meters between points

# Smoothing configuration
SMOOTHING_ENABLED = True  # Enable elevation smoothing
SMOOTHING_WINDOW = 20  # Number of points on each side for smoothing (2-5)
SMOOTHING_METHOD = 'gaussian'  # Options: 'gaussian', 'median', 'mean'

# Bridge/Tunnel filtering configuration
BRIDGE_TUNNEL_FILTER_ENABLED = True  # Enable bridge/tunnel detection and filtering
MAX_GRADIENT_PERCENT = 4.0  # Maximum realistic gradient for rail (4% is steep for rail)
OUTLIER_THRESHOLD_FT = 15.0  # Elevation jump threshold in feet for outlier detection
BRIDGE_TUNNEL_WINDOW_M = 50.0  # Window size in meters to check for structures

# API endpoints
USGS_TNM_API = "https://tnmaccess.nationalmap.gov/api/v1/products"
USGS_ELEVATION_API = "https://elevation.nationalmap.gov/arcgis/rest/services/3DEPElevation/ImageServer/identify"

# Output directory
OUTPUT_DIR = Path(INPUT_KMZ).parent
CACHE_DIR = OUTPUT_DIR / "Elevation_Cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = CACHE_DIR / "Downloaded_Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Search configuration
SEARCH_BUFFER_KM = 2.0  # Buffer around route in kilometers
LIDAR_SEARCH_OVERLAP = False  # Use overlapping search boxes for better coverage
LIDAR_SEARCH_BUFFER_DEG = 0.01  # Larger buffer for LiDAR searches (was 0.001)
OVERRIDE_CACHE = False  # If True, re-extract LiDAR and override conflicting cache entries
FORCE_LIDAR_RESEARCH = OVERRIDE_CACHE  # Force re-search for LiDAR even if cached

# LiDAR interpolation settings
LIDAR_INTERPOLATION_ENABLED = True  # Enable interpolation between LiDAR points
LIDAR_POINT_SPACING_M = 1.0  # Typical LiDAR point spacing in meters
LIDAR_INTERPOLATION_RADIUS_M = 10.0  # Search radius for nearby LiDAR points

# Manual LiDAR tiles (if automatic search misses some)
# Add URLs of specific LiDAR tiles that should be downloaded
MANUAL_LIDAR_TILES = [
    # Example: "https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/LPC/Projects/..."
    # Add any known LiDAR tile URLs here that aren't being found automatically
]

# LiDAR processing settings for actual data extraction
LIDAR_SEARCH_RADIUS_M = 20.0  # Search radius for nearest LiDAR points
LIDAR_MAX_POINTS = 10  # Maximum number of points to use for interpolation
LIDAR_IDW_POWER = 2.0  # Power parameter for inverse distance weighting

# Download settings
MAX_DOWNLOAD_WORKERS = 3  # Number of parallel downloads
CHUNK_SIZE = 8192  # Download chunk size
MAX_FILE_SIZE_MB = 500  # Maximum file size to download (MB)
AUTO_DELETE_CORRUPTED = True  # Automatically delete corrupted LiDAR files for re-download
AUTO_DELETE_CORRUPTED = True  # Automatically delete corrupted LiDAR files for re-download

# ==================== DATA STRUCTURES ====================

class SpatialElevationCache:
    """
    Lightweight spatial cache reader/updater for the pipeline.
    Uses the unified cache created by the converter tool.
    """
    
    def __init__(self, cache_dir: Path, tolerance_m: float = 0.1):
        self.cache_dir = cache_dir
        self.tolerance_m = tolerance_m
        
        # Cache files
        self.spatial_cache_file = cache_dir / "spatial_elevation_master.pkl.gz"
        self.spatial_index_file = cache_dir / "spatial_index.pkl.gz"
        
        # Load caches
        self.spatial_cache = self._load_spatial_cache()
        self.spatial_index = self._load_spatial_index()
        
        # Statistics for this session
        self.session_stats = {
            'lookups': 0,
            'hits': 0,
            'misses': 0,
            'points_added': 0
        }
        
        print(f"  Spatial cache loaded: {len(self.spatial_cache):,} elevation points")
        print(f"  Spatial tolerance: {self.tolerance_m}m")
    
    def _load_spatial_cache(self) -> Dict:
        """Load the unified spatial cache"""
        cache = {}
        
        # Check for single file or chunks
        if self.spatial_cache_file.exists():
            try:
                with gzip.open(self.spatial_cache_file, 'rb') as f:
                    cache = pickle.load(f)
            except Exception as e:
                print(f"  Warning: Could not load spatial cache: {e}")
        else:
            # Load from chunks
            chunk_files = sorted(self.cache_dir.glob("spatial_chunk_*.pkl.gz"))
            for chunk_file in chunk_files:
                try:
                    with gzip.open(chunk_file, 'rb') as f:
                        chunk_data = pickle.load(f)
                        cache.update(chunk_data)
                except Exception as e:
                    print(f"  Warning: Could not load {chunk_file.name}: {e}")
        
        return cache
    
    def _load_spatial_index(self) -> Dict:
        """Load the spatial index"""
        if self.spatial_index_file.exists():
            try:
                with gzip.open(self.spatial_index_file, 'rb') as f:
                    return pickle.load(f)
            except:
                pass
        return {}
    
    def _get_point_key(self, lat: float, lon: float) -> str:
        """Generate unique key for a coordinate"""
        return f"{lat:.7f}_{lon:.7f}"
    
    def _get_grid_key(self, lat: float, lon: float) -> str:
        """Get grid cell key for spatial indexing"""
        lat_m = lat * 111000
        lon_m = lon * 111000 * math.cos(math.radians(lat))
        grid_lat = int(lat_m / 10.0)  # 10m grid cells
        grid_lon = int(lon_m / 10.0)
        return f"{grid_lat},{grid_lon}"
    
    def _calculate_distance_m(self, lat1: float, lon1: float, 
                              lat2: float, lon2: float) -> float:
        """Calculate distance between two points in meters"""
        dlat = (lat2 - lat1) * 111000
        dlon = (lon2 - lon1) * 111000 * math.cos(math.radians((lat1 + lat2) / 2))
        return math.sqrt(dlat**2 + dlon**2)
    
    def find_cached_elevation(self, lat: float, lon: float) -> Optional[Dict]:
        """Find cached elevation within tolerance"""
        self.session_stats['lookups'] += 1
        
        # Check exact match first
        exact_key = self._get_point_key(lat, lon)
        if exact_key in self.spatial_cache:
            self.session_stats['hits'] += 1
            return self.spatial_cache[exact_key]
        
        # Check nearby grid cells
        center_key = self._get_grid_key(lat, lon)
        grid_lat, grid_lon = map(int, center_key.split(','))
        
        candidates = []
        for dlat in [-1, 0, 1]:
            for dlon in [-1, 0, 1]:
                cell_key = f"{grid_lat + dlat},{grid_lon + dlon}"
                if cell_key in self.spatial_index:
                    for point_key in self.spatial_index[cell_key]:
                        if point_key in self.spatial_cache:
                            cached = self.spatial_cache[point_key]
                            distance = self._calculate_distance_m(
                                lat, lon, cached['lat'], cached['lon']
                            )
                            if distance <= self.tolerance_m:
                                candidates.append((distance, cached))
        
        if candidates:
            candidates.sort(key=lambda x: x[0])
            self.session_stats['hits'] += 1
            return candidates[0][1]
        
        self.session_stats['misses'] += 1
        return None
    
    def process_points(self, points: List) -> Tuple[List, int]:
        """
        Process route points and use cached elevations where available.
        Returns (points, num_cache_hits)
        """
        cache_hits = 0
        
        for point in points:
            if point.elevation is None:
                cached = self.find_cached_elevation(point.lat, point.lon)
                if cached:
                    point.elevation = cached['elevation']
                    point.data_source = cached.get('data_source', 'cache')
                    point.source_priority = cached.get('source_priority', 10)
                    cache_hits += 1
        
        return points, cache_hits
    
    def add_extracted_elevations(self, points: List):
        """Add newly extracted elevations to cache"""
        added = 0
        for point in points:
            if point.elevation is not None and not hasattr(point, '_from_cache'):
                point_key = self._get_point_key(point.lat, point.lon)
                grid_key = self._get_grid_key(point.lat, point.lon)
                
                self.spatial_cache[point_key] = {
                    'lat': point.lat,
                    'lon': point.lon,
                    'elevation': point.elevation,
                    'data_source': point.data_source,
                    'source_priority': getattr(point, 'source_priority', 10),
                    'timestamp': time.time()
                }
                
                if grid_key not in self.spatial_index:
                    self.spatial_index[grid_key] = set()
                self.spatial_index[grid_key].add(point_key)
                
                added += 1
        
        self.session_stats['points_added'] = added
        
        if added > 0:
            self._save_cache()
            print(f"  Added {added:,} new elevations to spatial cache")
    
    def _save_cache(self):
        """Save updated cache"""
        try:
            # For simplicity, save as single file if under 100k points
            if len(self.spatial_cache) < 100000:
                with gzip.open(self.spatial_cache_file, 'wb') as f:
                    pickle.dump(self.spatial_cache, f, protocol=pickle.HIGHEST_PROTOCOL)
            else:
                # Save in chunks (implement if needed)
                pass
            
            # Save spatial index
            with gzip.open(self.spatial_index_file, 'wb') as f:
                pickle.dump(self.spatial_index, f, protocol=pickle.HIGHEST_PROTOCOL)
                
        except Exception as e:
            print(f"  Warning: Could not save spatial cache: {e}")
    
    def print_session_stats(self):
        """Print statistics for this session"""
        hit_rate = 0
        if self.session_stats['lookups'] > 0:
            hit_rate = (self.session_stats['hits'] / self.session_stats['lookups']) * 100
        
        print(f"\nSpatial Cache Statistics:")
        print(f"  Lookups: {self.session_stats['lookups']:,}")
        print(f"  Hits: {self.session_stats['hits']:,} ({hit_rate:.1f}%)")
        print(f"  Misses: {self.session_stats['misses']:,}")
        print(f"  New points added: {self.session_stats['points_added']:,}")

class RobustCache:
    """Base class for robust cache management with compression and error recovery"""
    
    def __init__(self, cache_path: Path, use_compression: bool = True, max_cache_size_mb: float = 500):
        """
        Initialize robust cache
        
        Args:
            cache_path: Path to cache file
            use_compression: Whether to use gzip compression
            max_cache_size_mb: Maximum cache size before triggering cleanup
        """
        self.cache_path = cache_path
        self.use_compression = use_compression
        self.max_cache_size_mb = max_cache_size_mb
        self.backup_dir = cache_path.parent / "cache_backups"
        self.backup_dir.mkdir(exist_ok=True)
    
    def _save_with_backup(self, data: Dict, description: str = "cache") -> bool:
        """
        Save data with automatic backup and error recovery
        
        Returns:
            True if save successful, False otherwise
        """
        # Create a temporary file first
        temp_path = self.cache_path.with_suffix('.tmp')
        
        try:
            # Save to temporary file
            if self.use_compression:
                with gzip.open(temp_path, 'wb') as f:
                    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
            else:
                with open(temp_path, 'wb') as f:
                    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
            
            # Check file size
            file_size_mb = temp_path.stat().st_size / (1024 * 1024)
            if file_size_mb > self.max_cache_size_mb:
                print(f"   Warning: {description} size ({file_size_mb:.1f}MB) exceeds limit ({self.max_cache_size_mb}MB)")
                print(f"   Consider clearing old data or increasing limit")
            
            # Backup existing cache if it exists
            if self.cache_path.exists():
                backup_path = self.backup_dir / f"{self.cache_path.stem}_{datetime.now():%Y%m%d_%H%M%S}{self.cache_path.suffix}"
                try:
                    shutil.move(str(self.cache_path), str(backup_path))
                    # Keep only last 3 backups
                    self._cleanup_old_backups()
                except:
                    pass  # Backup failed, but we can still continue
            
            # Move temp file to actual cache file
            shutil.move(str(temp_path), str(self.cache_path))
            
            print(f"   Saved {description} ({file_size_mb:.1f}MB)")
            return True
            
        except Exception as e:
            print(f"   Error saving {description}: {e}")
            if temp_path.exists():
                temp_path.unlink()
            return False
    
    def _load_with_recovery(self, description: str = "cache") -> Optional[Dict]:
        """
        Load data with automatic error recovery
        
        Returns:
            Loaded data or None if load fails
        """
        # Try loading the main cache file
        if self.cache_path.exists():
            try:
                if self.use_compression:
                    with gzip.open(self.cache_path, 'rb') as f:
                        data = pickle.load(f)
                else:
                    with open(self.cache_path, 'rb') as f:
                        data = pickle.load(f)
                
                file_size_mb = self.cache_path.stat().st_size / (1024 * 1024)
                print(f"   Loaded {description} ({file_size_mb:.1f}MB)")
                return data
                
            except Exception as e:
                print(f"   Error loading {description}: {e}")
                
                # Try to recover from backup
                backups = sorted(self.backup_dir.glob(f"{self.cache_path.stem}*{self.cache_path.suffix}"))
                if backups:
                    print(f"   Attempting recovery from {len(backups)} backup(s)...")
                    
                    for backup_path in reversed(backups):  # Try newest first
                        try:
                            if self.use_compression:
                                with gzip.open(backup_path, 'rb') as f:
                                    data = pickle.load(f)
                            else:
                                with open(backup_path, 'rb') as f:
                                    data = pickle.load(f)
                            
                            print(f"   Successfully recovered from backup: {backup_path.name}")
                            
                            # Save corrupted file for debugging
                            corrupted_path = self.cache_path.with_suffix(f'.corrupted_{datetime.now():%Y%m%d_%H%M%S}')
                            try:
                                shutil.move(str(self.cache_path), str(corrupted_path))
                                print(f"   Corrupted cache saved as: {corrupted_path.name}")
                            except:
                                pass
                            
                            # Restore from backup
                            shutil.copy2(str(backup_path), str(self.cache_path))
                            return data
                            
                        except Exception as backup_error:
                            print(f"   Backup {backup_path.name} also corrupted: {backup_error}")
                            continue
                
                print(f"   No valid backups found, starting fresh")
                
                # Move corrupted file
                try:
                    corrupted_path = self.cache_path.with_suffix(f'.corrupted_{datetime.now():%Y%m%d_%H%M%S}')
                    shutil.move(str(self.cache_path), str(corrupted_path))
                except:
                    pass
        
        return None
    
    def _cleanup_old_backups(self, keep_count: int = 3):
        """Keep only the most recent backups"""
        backups = sorted(self.backup_dir.glob(f"{self.cache_path.stem}*{self.cache_path.suffix}"))
        if len(backups) > keep_count:
            for old_backup in backups[:-keep_count]:
                try:
                    old_backup.unlink()
                except:
                    pass

@dataclass
class RoutePoint:
    """Represents an interpolated point along the route"""
    index: int
    lat: float
    lon: float
    distance_m: float
    elevation: Optional[float] = None  # Will store in feet
    data_source: Optional[str] = None
    source_priority: int = 999
    is_interpolated: bool = False

    def location_hash(self, precision: int = 6) -> str:
        """Generate a hash based on location (lat/lon) for cache matching"""
        # Round to specified decimal precision to handle floating point differences
        lat_rounded = round(self.lat, precision)
        lon_rounded = round(self.lon, precision)
        return f"{lat_rounded:.{precision}f},{lon_rounded:.{precision}f}"

class SmartInterpolationCache:
    """Manages intelligent caching with chunking for very large datasets"""
    
    def __init__(self, cache_dir: Path, use_chunking: bool = True, chunk_size: int = 50000):
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(exist_ok=True)
        
        # Cache files with compression
        self.cache_file = cache_dir / "smart_interpolation_cache.pkl.gz"
        self.location_index_file = cache_dir / "location_index.pkl.gz"
        self.global_cache_file = cache_dir / "global_elevation_cache.pkl.gz"
        
        # Settings
        self.use_chunking = use_chunking
        self.chunk_size = chunk_size
        
        # Initialize cache managers
        self.cache_manager = RobustCache(self.cache_file, use_compression=True)
        self.location_manager = RobustCache(self.location_index_file, use_compression=True)
        self.global_manager = RobustCache(self.global_cache_file, use_compression=True, max_cache_size_mb=1000)
        
        # Load caches
        self.cache_data = self._load_cache()
        self.location_index = self._load_location_index()
        self.global_cache = self._load_global_cache()
    
    def _load_cache(self) -> Dict:
        """Load the main cache containing interpolated points and their data"""
        data = self.cache_manager._load_with_recovery("smart interpolation cache")
        if data:
            print(f"   Smart cache contains {len(data.get('points', {}))} cached points")
            return data
        return {'points': {}, 'kmz_hash': None, 'last_updated': None}
    
    def _load_location_index(self) -> Dict[str, Dict]:
        """Load the location index that maps locations to cached point data"""
        data = self.location_manager._load_with_recovery("location index")
        if data:
            print(f"   Location index contains {len(data)} unique locations")
            return data
        return {}
    
    def _load_global_cache(self) -> Dict[str, Dict]:
        """Load global cache that persists across all files with chunking support"""
        if self.use_chunking:
            # Try to load chunked cache
            return self._load_chunked_global_cache()
        else:
            data = self.global_manager._load_with_recovery("global cache")
            if data:
                print(f"   Global cache contains {len(data)} locations from all previous runs")
                return data
            return {}
    
    def _load_chunked_global_cache(self) -> Dict[str, Dict]:
        """Load global cache from multiple chunk files"""
        global_cache = {}
        chunk_files = sorted(self.cache_dir.glob("global_chunk_*.pkl.gz"))
        
        if chunk_files:
            print(f"   Loading global cache from {len(chunk_files)} chunks...")
            for chunk_file in chunk_files:
                chunk_manager = RobustCache(chunk_file, use_compression=True)
                chunk_data = chunk_manager._load_with_recovery(f"chunk {chunk_file.stem}")
                if chunk_data:
                    global_cache.update(chunk_data)
            
            print(f"   Global cache loaded: {len(global_cache)} total locations")
        
        return global_cache
    
    def _save_cache(self):
        """Save all caches with chunking for global cache if needed"""
        # Save main cache and location index
        self.cache_data['last_updated'] = time.time()
        self.cache_manager._save_with_backup(self.cache_data, "smart interpolation cache")
        self.location_manager._save_with_backup(self.location_index, "location index")
        
        # Save global cache with chunking if needed
        if self.use_chunking and len(self.global_cache) > self.chunk_size:
            self._save_chunked_global_cache()
        else:
            self.global_manager._save_with_backup(self.global_cache, "global cache")
    
    def _save_chunked_global_cache(self):
        """Save global cache in chunks for better performance with large datasets"""
        # Clear old chunks
        old_chunks = self.cache_dir.glob("global_chunk_*.pkl.gz")
        for old_chunk in old_chunks:
            try:
                old_chunk.unlink()
            except:
                pass
        
        # Save in chunks
        items = list(self.global_cache.items())
        chunk_count = (len(items) + self.chunk_size - 1) // self.chunk_size
        
        print(f"   Saving global cache in {chunk_count} chunks...")
        
        for i in range(chunk_count):
            start_idx = i * self.chunk_size
            end_idx = min((i + 1) * self.chunk_size, len(items))
            chunk_data = dict(items[start_idx:end_idx])
            
            chunk_file = self.cache_dir / f"global_chunk_{i:04d}.pkl.gz"
            chunk_manager = RobustCache(chunk_file, use_compression=True)
            chunk_manager._save_with_backup(chunk_data, f"global chunk {i+1}/{chunk_count}")
    
    def get_kmz_hash(self, kmz_path: str) -> str:
        """Calculate hash of KMZ file to detect changes"""
        with open(kmz_path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    
    def match_points_with_cache(self, new_points: List, kmz_path: str) -> Tuple[List, Dict[int, int]]:
        """
        Match new interpolated points with cached points based on location.
        Now includes global cache lookup for cross-file support.
        """
        kmz_hash = self.get_kmz_hash(kmz_path)
        index_mapping = {}
        matched_count = 0
        global_cache_hits = 0
        new_locations_count = 0
        reused_elevations = 0
        
        print("\n" + "="*60)
        print("SMART CACHE MATCHING")
        print("="*60)
        
        # Check if KMZ has changed
        if self.cache_data.get('kmz_hash') != kmz_hash:
            print(f"  KMZ file has changed, performing smart location matching...")
        else:
            print(f"  KMZ file unchanged, checking for index shifts and cached data...")
        
        # Match points with cached data
        for point in new_points:
            loc_hash = point.location_hash()
            
            # First check local cache
            if loc_hash in self.location_index:
                cached_data = self.location_index[loc_hash]
                matched_count += 1
                
                if 'old_index' in cached_data:
                    index_mapping[point.index] = cached_data['old_index']
                
                if cached_data.get('elevation') is not None:
                    point.elevation = cached_data['elevation']
                    point.data_source = cached_data.get('data_source')
                    point.source_priority = cached_data.get('source_priority', 999)
                    point.is_interpolated = cached_data.get('is_interpolated', False)
                    reused_elevations += 1
            
            # Check global cache
            elif loc_hash in self.global_cache:
                cached_data = self.global_cache[loc_hash]
                global_cache_hits += 1
                
                if cached_data.get('elevation') is not None:
                    point.elevation = cached_data['elevation']
                    point.data_source = cached_data.get('data_source', 'Global Cache')
                    point.source_priority = cached_data.get('source_priority', 999)
                    point.is_interpolated = cached_data.get('is_interpolated', False)
                    reused_elevations += 1
            else:
                new_locations_count += 1
        
        # Report results
        print(f"\n  Cache matching results:")
        print(f"    Matched locations (local): {matched_count}")
        print(f"    Matched locations (global): {global_cache_hits}")
        print(f"    New locations: {new_locations_count}")
        print(f"    Reused elevations: {reused_elevations}")
        
        if reused_elevations > 0:
            print(f"    Cache hit rate: {reused_elevations/len(new_points)*100:.1f}%")
        
        # Update cache with new hash
        self.cache_data['kmz_hash'] = kmz_hash
        self._save_cache()
        
        return new_points, index_mapping
    
    def _update_cache(self, points: List, kmz_hash: str):
        """Update both local and global cache with new point data"""
        # Update local cache
        self.cache_data = {
            'points': {},
            'kmz_hash': kmz_hash,
            'last_updated': time.time()
        }
        
        # Rebuild location index
        self.location_index = {}
        
        for point in points:
            loc_hash = point.location_hash()
            
            point_data = {
                'old_index': point.index,
                'elevation': point.elevation,
                'data_source': point.data_source,
                'source_priority': point.source_priority,
                'is_interpolated': point.is_interpolated,
                'lat': point.lat,
                'lon': point.lon,
                'distance_m': point.distance_m
            }
            
            # Store in local location index
            self.location_index[loc_hash] = point_data
            
            # Update global cache
            if point.elevation is not None:
                self.global_cache[loc_hash] = point_data
            
            # Store in main cache by index
            self.cache_data['points'][point.index] = {
                'location_hash': loc_hash,
                'elevation': point.elevation,
                'data_source': point.data_source
            }
        
        print(f"   Updated caches: {len(points)} points in local, {len(self.global_cache)} in global")
        self._save_cache()
    
    def scan_existing_elevation_files(self, output_dir: Path, rebuild_from_csv: bool = False):
        """
        Scan existing elevation files and update global cache
        
        Args:
            output_dir: Directory containing elevation CSV files
            rebuild_from_csv: If True, rebuild cache from CSV files. If False (default),
                            only add CSV data that's not already in cache
        """
        import pandas as pd
        
        print("\n" + "="*60)
        print("SCANNING EXISTING ELEVATION FILES FOR CACHE")
        print("="*60)
        
        # First, report on existing pickle cache
        existing_cache_size = len(self.global_cache)
        if existing_cache_size > 0:
            print(f"  Existing global cache contains: {existing_cache_size:,} locations")
            
            if not rebuild_from_csv:
                print("  Preserving existing cache data")
        
        # Find CSV files to potentially add to cache
        csv_files = list(output_dir.glob("*_elevations.csv")) + list(output_dir.glob("*_elevation_data.csv"))
        
        if not csv_files:
            print("  No CSV elevation files found in output directory")
            if existing_cache_size > 0:
                print(f"  Cache still contains {existing_cache_size:,} locations from previous runs")
            return
        
        print(f"  Found {len(csv_files)} CSV elevation files to scan")
        
        # If rebuilding, clear the cache first
        if rebuild_from_csv and existing_cache_size > 0:
            print(f"  Clearing existing cache ({existing_cache_size:,} locations) for rebuild")
            self.global_cache.clear()
        
        total_added = 0
        total_skipped = 0
        
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                
                if 'Elevation_ft' not in df.columns:
                    print(f"    Skipping {csv_file.name}: No Elevation_ft column")
                    continue
                
                file_added = 0
                file_skipped = 0
                
                for _, row in df.iterrows():
                    if pd.notna(row.get('Elevation_ft')):
                        lat = row.get('Latitude')
                        lon = row.get('Longitude')
                        
                        if lat is not None and lon is not None:
                            precision = 6
                            loc_hash = f"{lat:.{precision}f}_{lon:.{precision}f}"
                            
                            # Check if already in cache
                            if loc_hash in self.global_cache and not rebuild_from_csv:
                                file_skipped += 1
                                continue
                            
                            # Convert elevation from feet to meters for storage
                            elevation_m = row['Elevation_ft'] * 0.3048
                            
                            # Add or update in global cache
                            self.global_cache[loc_hash] = {
                                'lat': lat,
                                'lon': lon,
                                'elevation': elevation_m,
                                'data_type': row.get('DataSource', 'unknown'),
                                'source_file': csv_file.name,
                                'timestamp': time.time()
                            }
                            file_added += 1
                
                if file_added > 0:
                    print(f"    {csv_file.name}: added {file_added:,} elevations")
                if file_skipped > 0:
                    print(f"      (skipped {file_skipped:,} already cached)")
                
                total_added += file_added
                total_skipped += file_skipped
                
            except Exception as e:
                print(f"    Error reading {csv_file.name}: {e}")
                continue
        
        # Report summary
        print(f"\n  Summary:")
        print(f"    Total new elevations added: {total_added:,}")
        if total_skipped > 0:
            print(f"    Total duplicates skipped: {total_skipped:,}")
        print(f"    Global cache now contains: {len(self.global_cache):,} unique locations")
        
        # Save the updated cache
        if total_added > 0:
            print("  Saving updated global cache...")
        self._save_cache()


@dataclass
class ElevationTile:
    """Represents an elevation data tile"""
    name: str
    bbox: Tuple[float, float, float, float]  # min_lon, min_lat, max_lon, max_lat
    url: Optional[str]
    resolution: float
    data_type: str
    priority: int
    format: Optional[str] = None
    covered_points: Set[int] = field(default_factory=set)
    interpolated_points: Set[int] = field(default_factory=set)
    local_path: Optional[Path] = None

@dataclass
class BridgeTunnelSegment:
    """Represents a bridge or tunnel segment from KMZ"""
    name: str
    coordinates: List[Tuple[float, float]]  # List of (lat, lon) pairs
    start_lat: float
    start_lon: float
    end_lat: float
    end_lon: float
    
# ==================== UTILITY FUNCTIONS ====================

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance between two points on Earth in meters"""
    R = 6371000  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def extract_bridge_tunnel_segments(kmz_path: str) -> List[BridgeTunnelSegment]:
    """Extract bridge/tunnel segments from a KMZ file"""
    if not kmz_path or not os.path.exists(kmz_path):
        return []
    
    print(f"\nExtracting bridge/tunnel segments from {os.path.basename(kmz_path)}...")
    
    segments = []
    
    try:
        with tempfile.TemporaryDirectory() as temp_dir:
            with zipfile.ZipFile(kmz_path, 'r') as kmz:
                kmz.extractall(temp_dir)
                
                kml_file = None
                for file in os.listdir(temp_dir):
                    if file.endswith('.kml'):
                        kml_file = os.path.join(temp_dir, file)
                        break
                
                if not kml_file:
                    print("  Warning: No KML file found in bridge/tunnel KMZ")
                    return []
                
                tree = ET.parse(kml_file)
                root = tree.getroot()
        
        ns = {'kml': 'http://www.opengis.net/kml/2.2'}
        if not root.findall('.//kml:coordinates', ns):
            ns = {}
        
        # Find all Placemarks that contain LineStrings
        if ns:
            placemarks = root.findall('.//kml:Placemark', ns)
        else:
            placemarks = root.findall('.//Placemark')
        
        for placemark in placemarks:
            # Get the name of this segment if available
            if ns:
                name_elem = placemark.find('kml:name', ns)
                linestring = placemark.find('.//kml:LineString', ns)
            else:
                name_elem = placemark.find('name')
                linestring = placemark.find('.//LineString')
            
            name = name_elem.text if name_elem is not None else "Unnamed Bridge/Tunnel"
            
            if linestring is not None:
                if ns:
                    coords_elem = linestring.find('kml:coordinates', ns)
                else:
                    coords_elem = linestring.find('coordinates')
                
                if coords_elem is not None:
                    segment_coordinates = []
                    coords_text = coords_elem.text.strip()
                    for coord in coords_text.split():
                        parts = coord.split(',')
                        if len(parts) >= 2:
                            lon = float(parts[0])
                            lat = float(parts[1])
                            segment_coordinates.append((lat, lon))
                    
                    if len(segment_coordinates) >= 2:
                        segment = BridgeTunnelSegment(
                            name=name,
                            coordinates=segment_coordinates,
                            start_lat=segment_coordinates[0][0],
                            start_lon=segment_coordinates[0][1],
                            end_lat=segment_coordinates[-1][0],
                            end_lon=segment_coordinates[-1][1]
                        )
                        segments.append(segment)
                        print(f"  Found bridge/tunnel segment '{name}' with {len(segment_coordinates)} points")
        
        print(f"  Total bridge/tunnel segments found: {len(segments)}")
        
    except Exception as e:
        print(f"  Error extracting bridge/tunnel segments: {e}")
    
    return segments

def extract_manual_elevation_waypoints(kmz_path: str) -> List[Dict]:
    """Extract waypoints with manual elevation specifications from their names"""
    if not kmz_path or not os.path.exists(kmz_path):
        return []
    
    print(f"\nExtracting manual elevation waypoints from {os.path.basename(kmz_path)}...")
    elevation_waypoints = []
    
    # Pattern to detect and extract elevation from waypoint names
    elevation_pattern = re.compile(r'(\d+\.?\d*)\s*FT', re.IGNORECASE)
    
    with tempfile.TemporaryDirectory() as temp_dir:
        with zipfile.ZipFile(kmz_path, 'r') as kmz:
            kmz.extractall(temp_dir)
            
            kml_file = None
            for file in os.listdir(temp_dir):
                if file.endswith('.kml'):
                    kml_file = os.path.join(temp_dir, file)
                    break
            
            if not kml_file:
                return []
            
            tree = ET.parse(kml_file)
            root = tree.getroot()
    
    # Check for namespace
    ns = {'kml': 'http://www.opengis.net/kml/2.2'}
    if not root.findall('.//kml:Point', ns):
        ns = {}
    
    # Extract Point placemarks with elevation in their names
    if ns:
        placemarks = root.findall('.//kml:Placemark', ns)
    else:
        placemarks = root.findall('.//Placemark')
    
    for placemark in placemarks:
        if ns:
            point = placemark.find('.//kml:Point', ns)
            name_elem = placemark.find('kml:name', ns)
        else:
            point = placemark.find('.//Point')
            name_elem = placemark.find('name')
        
        if point is not None and name_elem is not None:
            waypoint_name = name_elem.text
            
            # Check if this waypoint has an elevation specification
            elevation_match = elevation_pattern.search(waypoint_name)
            if elevation_match:
                elevation_ft = float(elevation_match.group(1))
                
                if ns:
                    coords_elem = point.find('kml:coordinates', ns)
                else:
                    coords_elem = point.find('coordinates')
                
                if coords_elem is not None:
                    coords_text = coords_elem.text.strip()
                    parts = coords_text.split(',')
                    if len(parts) >= 2:
                        elevation_waypoints.append({
                            'name': waypoint_name,
                            'latitude': float(parts[1]),
                            'longitude': float(parts[0]),
                            'elevation_ft': elevation_ft
                        })
                        print(f"  Found elevation waypoint '{waypoint_name}' at {elevation_ft} ft")
    
    print(f"  Total manual elevation waypoints found: {len(elevation_waypoints)}")
    return elevation_waypoints

def apply_manual_elevation_waypoints(points: List[RoutePoint], elevation_waypoints: List[Dict]) -> List[RoutePoint]:
    """Apply manual elevation values from waypoints to the nearest route points"""
    if not elevation_waypoints:
        return points
    
    print(f"\nApplying {len(elevation_waypoints)} manual elevation waypoints...")
    
    for waypoint in elevation_waypoints:
        # Find the nearest route point
        min_distance = float('inf')
        nearest_idx = None
        
        for i, point in enumerate(points):
            distance = haversine_distance(
                point.lat, point.lon,
                waypoint['latitude'], waypoint['longitude']
            )
            
            if distance < min_distance:
                min_distance = distance
                nearest_idx = i
        
        # Apply elevation if within reasonable distance (50m threshold)
        if nearest_idx is not None and min_distance < 50:
            old_elevation = points[nearest_idx].elevation
            points[nearest_idx].elevation = waypoint['elevation_ft']
            points[nearest_idx].data_source = "Manual Waypoint"
            points[nearest_idx].source_priority = 0  # Highest priority
            
            print(f"  Applied '{waypoint['name']}' elevation {waypoint['elevation_ft']} ft")
            print(f"    to point {nearest_idx} (distance: {min_distance:.1f}m)")
            if old_elevation is not None:
                print(f"    replaced elevation: {old_elevation:.1f} ft")
        else:
            print(f"  WARNING: No route point within 50m for waypoint '{waypoint['name']}'")
            if nearest_idx is not None:
                print(f"    Nearest point is {min_distance:.1f}m away")
    
    return points

def point_in_segment(point: RoutePoint, segment: BridgeTunnelSegment, buffer_m: float = 20.0) -> bool:
    """Check if a point is within a bridge/tunnel segment with buffer"""
    # Check if point is close to any line segment of the bridge/tunnel
    for i in range(len(segment.coordinates) - 1):
        lat1, lon1 = segment.coordinates[i]
        lat2, lon2 = segment.coordinates[i + 1]
        
        # Calculate distance from point to line segment
        # Using simplified approach: check if point is within buffer distance of segment endpoints
        # or the line between them
        
        # Distance to endpoints
        dist_to_start = haversine_distance(point.lat, point.lon, lat1, lon1)
        dist_to_end = haversine_distance(point.lat, point.lon, lat2, lon2)
        
        if dist_to_start <= buffer_m or dist_to_end <= buffer_m:
            return True
        
        # Approximate distance to line segment (simplified)
        segment_length = haversine_distance(lat1, lon1, lat2, lon2)
        if segment_length > 0:
            # Check if point is roughly between the endpoints
            total_dist = dist_to_start + dist_to_end
            if total_dist <= segment_length + 2 * buffer_m:
                return True
    
    return False

def extract_coordinates_from_kmz_with_path_fixing(kmz_path: str) -> List[Tuple[float, float]]:
    """Extract coordinates from KMZ file with automatic path reversal detection and fixing"""
    print(f"Extracting coordinates from {os.path.basename(kmz_path)}...")
    
    with tempfile.TemporaryDirectory() as temp_dir:
        with zipfile.ZipFile(kmz_path, 'r') as kmz:
            kmz.extractall(temp_dir)
            
            kml_file = None
            for file in os.listdir(temp_dir):
                if file.endswith('.kml'):
                    kml_file = os.path.join(temp_dir, file)
                    break
            
            if not kml_file:
                raise ValueError("No KML file found in KMZ")
            
            tree = ET.parse(kml_file)
            root = tree.getroot()
    
    ns = {'kml': 'http://www.opengis.net/kml/2.2'}
    if not root.findall('.//kml:coordinates', ns):
        ns = {}
    
    all_paths = []
    path_names = []
    
    # Find all Placemarks that contain LineStrings
    if ns:
        placemarks = root.findall('.//kml:Placemark', ns)
    else:
        placemarks = root.findall('.//Placemark')
    
    for placemark in placemarks:
        # Get the name of this path if available
        if ns:
            name_elem = placemark.find('kml:name', ns)
            linestring = placemark.find('.//kml:LineString', ns)
        else:
            name_elem = placemark.find('name')
            linestring = placemark.find('.//LineString')
        
        name = name_elem.text if name_elem is not None else "Unnamed"
        
        if linestring is not None:
            if ns:
                coords_elem = linestring.find('kml:coordinates', ns)
            else:
                coords_elem = linestring.find('coordinates')
            
            if coords_elem is not None:
                path_coordinates = []
                coords_text = coords_elem.text.strip()
                for coord in coords_text.split():
                    parts = coord.split(',')
                    if len(parts) >= 2:
                        lon = float(parts[0])
                        lat = float(parts[1])
                        path_coordinates.append((lat, lon))
                
                if path_coordinates:
                    all_paths.append(path_coordinates)
                    path_names.append(name)
                    print(f"  Found path '{name}' with {len(path_coordinates)} points")
    
    # If no placemarks found, fall back to looking for LineStrings directly
    if not all_paths:
        if ns:
            linestrings = root.findall('.//kml:LineString', ns)
        else:
            linestrings = root.findall('.//LineString')
        
        for i, linestring in enumerate(linestrings):
            if ns:
                coords_elem = linestring.find('.//kml:coordinates', ns)
            else:
                coords_elem = linestring.find('.//coordinates')
            
            if coords_elem is not None:
                path_coordinates = []
                coords_text = coords_elem.text.strip()
                for coord in coords_text.split():
                    parts = coord.split(',')
                    if len(parts) >= 2:
                        lon = float(parts[0])
                        lat = float(parts[1])
                        path_coordinates.append((lat, lon))
                
                if path_coordinates:
                    all_paths.append(path_coordinates)
                    path_names.append(f"Path {i+1}")
                    print(f"  Found path {i+1} with {len(path_coordinates)} points")
    
    if not all_paths:
        raise ValueError("No coordinate paths found in KMZ")
    
    print(f"\n  Total paths found: {len(all_paths)}")
    
    # Calculate statistics for each path
    print("\n  Path details:")
    path_info = []
    for i, (path, name) in enumerate(zip(all_paths, path_names)):
        if len(path) > 1:
            # Calculate path length
            total_dist = 0
            for j in range(1, len(path)):
                dist = haversine_distance(path[j-1][0], path[j-1][1], path[j][0], path[j][1])
                total_dist += dist
            
            info = {
                'index': i,
                'name': name,
                'points': len(path),
                'length_km': total_dist/1000,
                'start': path[0],
                'end': path[-1],
                'path': path
            }
            path_info.append(info)
            
            print(f"    Path {i+1} '{name}':")
            print(f"      Points: {len(path)}")
            print(f"      Length: {total_dist/1000:.2f} km")
            print(f"      Start: ({path[0][0]:.6f}, {path[0][1]:.6f})")
            print(f"      End:   ({path[-1][0]:.6f}, {path[-1][1]:.6f})")
    
    # FIXED: Path Reversal Detection and Fixing
    if len(all_paths) > 1:
        print("\n  Checking path continuity and detecting reversals...")
        
        # Create a working copy of paths that we'll modify
        working_paths = [path[:] for path in all_paths]  # Deep copy
        
        # Process paths sequentially, checking best connection for each pair
        for i in range(len(working_paths) - 1):
            current_path = working_paths[i]
            next_path = working_paths[i + 1]
            
            # Calculate all possible connection distances
            connections = {
                'normal': haversine_distance(
                    current_path[-1][0], current_path[-1][1],
                    next_path[0][0], next_path[0][1]
                ),
                'reverse_next': haversine_distance(
                    current_path[-1][0], current_path[-1][1],
                    next_path[-1][0], next_path[-1][1]
                ),
                'reverse_current': haversine_distance(
                    current_path[0][0], current_path[0][1],
                    next_path[0][0], next_path[0][1]
                ),
                'reverse_both': haversine_distance(
                    current_path[0][0], current_path[0][1],
                    next_path[-1][0], next_path[-1][1]
                )
            }
            
            print(f"\n    Connection analysis for Path {i+1} -> Path {i+2}:")
            for conn_type, distance in connections.items():
                print(f"      {conn_type}: {distance:.1f}m")
            
            # Find the best connection (minimum distance)
            best_connection = min(connections.items(), key=lambda x: x[1])
            best_type, best_distance = best_connection
            
            print(f"      BEST: {best_type} ({best_distance:.1f}m)")
            
            # Apply the necessary reversals based on best connection
            if best_type == 'reverse_next':
                working_paths[i + 1] = list(reversed(working_paths[i + 1]))
                print(f"      → Reversing Path {i+2}")
            elif best_type == 'reverse_current':
                working_paths[i] = list(reversed(working_paths[i]))
                print(f"      → Reversing Path {i+1}")
                # Need to recalculate if we reversed the current path and i > 0
                if i > 0:
                    # Check if this breaks the previous connection
                    prev_gap = haversine_distance(
                        working_paths[i-1][-1][0], working_paths[i-1][-1][1],
                        working_paths[i][0][0], working_paths[i][0][1]
                    )
                    if prev_gap > 1000:  # If we broke the previous connection
                        print(f"      WARNING: Reversing Path {i+1} breaks connection to Path {i}")
                        print(f"      Consider manual review of path ordering")
            elif best_type == 'reverse_both':
                working_paths[i] = list(reversed(working_paths[i]))
                working_paths[i + 1] = list(reversed(working_paths[i + 1]))
                print(f"      → Reversing both Path {i+1} and Path {i+2}")
                # Check previous connection if needed
                if i > 0:
                    prev_gap = haversine_distance(
                        working_paths[i-1][-1][0], working_paths[i-1][-1][1],
                        working_paths[i][0][0], working_paths[i][0][1]
                    )
                    if prev_gap > 1000:
                        print(f"      WARNING: Reversing Path {i+1} breaks connection to Path {i}")
            # else: normal - no reversal needed
            
            # Warn if gap is still large after optimal connection
            if best_distance > 1000:
                print(f"      ⚠ Large gap remains ({best_distance:.1f}m) even with best connection")
                if best_distance > 50000:  # > 50km gap
                    print(f"      ⚠ VERY LARGE GAP - paths might be in wrong order or from different routes")
        
        # Replace original paths with working paths
        all_paths = working_paths
        
        # Show final connections after all reversals
        print("\n  Final path connections after reversal:")
        total_gap = 0
        for i in range(len(all_paths) - 1):
            end_of_current = all_paths[i][-1]
            start_of_next = all_paths[i+1][0]
            distance = haversine_distance(end_of_current[0], end_of_current[1], 
                                        start_of_next[0], start_of_next[1])
            total_gap += distance
            
            status = "✓" if distance < 100 else ("⚠" if distance < 1000 else "✗")
            print(f"    {status} Gap between path {i+1} and path {i+2}: {distance:.1f}m")
        
        avg_gap = total_gap / (len(all_paths) - 1)
        print(f"\n  Average gap: {avg_gap:.1f}m")
        
        if avg_gap > 10000:
            print("\n  ⚠ WARNING: Large average gap suggests paths may be:")
            print("    - From different routes")
            print("    - In the wrong order")
            print("    - Missing connecting segments")
    
    # Determine how to handle multiple paths based on configuration
    if len(all_paths) == 1:
        coordinates = all_paths[0]
        print(f"\n  Using single path with {len(coordinates)} points")
    else:
        print(f"\n  Multiple paths detected. Using '{PATH_HANDLING}' strategy:")
        
        if PATH_HANDLING == 'connect' or PATH_HANDLING == 'all':
            # Connect all paths (they should now be properly oriented)
            coordinates = []
            for i, path in enumerate(all_paths):
                coordinates.extend(path)
            print(f"  Connected {len(all_paths)} paths: total {len(coordinates)} points")
            
        elif PATH_HANDLING == 'longest':
            # Use the longest path
            path_lengths = []
            for path in all_paths:
                if len(path) > 1:
                    total_dist = sum(haversine_distance(path[j-1][0], path[j-1][1], 
                                                       path[j][0], path[j][1]) 
                                   for j in range(1, len(path)))
                    path_lengths.append(total_dist)
                else:
                    path_lengths.append(0)
            
            longest_path_idx = max(range(len(all_paths)), key=lambda i: path_lengths[i])
            coordinates = all_paths[longest_path_idx]
            print(f"  Using longest path '{path_names[longest_path_idx]}' ({path_lengths[longest_path_idx]/1000:.2f} km)")
            
        elif PATH_HANDLING == 'first':
            coordinates = all_paths[0]
            print(f"  Using first path '{path_names[0]}' with {len(coordinates)} points")
            
        elif PATH_HANDLING == 'last':
            coordinates = all_paths[-1]
            print(f"  Using last path '{path_names[-1]}' with {len(coordinates)} points")
            
        else:
            # Default to connect
            print(f"  Unknown strategy '{PATH_HANDLING}', defaulting to 'connect'")
            coordinates = []
            for path in all_paths:
                coordinates.extend(path)
            print(f"  Connected all paths: total {len(coordinates)} points")
    
    print(f"\n  Final extraction: {len(coordinates)} coordinate points")
    
    # Remove any duplicate consecutive points
    cleaned_coordinates = []
    for i, coord in enumerate(coordinates):
        if i == 0 or coord != coordinates[i-1]:
            cleaned_coordinates.append(coord)
        else:
            print(f"  Removed duplicate point at index {i}")
    
    if len(cleaned_coordinates) < len(coordinates):
        print(f"  Cleaned: {len(coordinates)} → {len(cleaned_coordinates)} points")
        coordinates = cleaned_coordinates
    
    return coordinates

def interpolate_route(coordinates: List[Tuple[float, float]], spacing_m: float) -> List[RoutePoint]:
    """Interpolate route points at regular spacing"""
    print(f"\nInterpolating route to {spacing_m}m spacing...")
    
    interpolated_points = []
    total_distance = 0
    point_index = 0
    
    for i in range(len(coordinates) - 1):
        lat1, lon1 = coordinates[i]
        lat2, lon2 = coordinates[i + 1]
        
        segment_distance = haversine_distance(lat1, lon1, lat2, lon2)
        
        if segment_distance < spacing_m:
            interpolated_points.append(RoutePoint(
                index=point_index,
                lat=lat1,
                lon=lon1,
                distance_m=total_distance
            ))
            point_index += 1
        else:
            num_points = int(segment_distance / spacing_m)
            for j in range(num_points):
                fraction = j / num_points
                lat = lat1 + fraction * (lat2 - lat1)
                lon = lon1 + fraction * (lon2 - lon1)
                
                interpolated_points.append(RoutePoint(
                    index=point_index,
                    lat=lat,
                    lon=lon,
                    distance_m=total_distance + j * spacing_m
                ))
                point_index += 1
        
        total_distance += segment_distance
    
    if coordinates:
        lat, lon = coordinates[-1]
        interpolated_points.append(RoutePoint(
            index=point_index,
            lat=lat,
            lon=lon,
            distance_m=total_distance
        ))
    
    print(f"  Created {len(interpolated_points)} interpolated points")
    print(f"  Total route distance: {total_distance/1000:.2f} km")
    
    return interpolated_points

def interpolate_route_with_smart_cache(
    coordinates: List[Tuple[float, float]], 
    spacing_m: float,
    cache_dir: Path,
    kmz_path: str
) -> List[RoutePoint]:
    """
    Interpolate route points with smart caching that handles index changes.
    """
    print(f"\nInterpolating route to {spacing_m}m spacing with SMART CACHING...")
    
    # First, do the regular interpolation
    interpolated_points = []
    total_distance = 0
    point_index = 0
    
    for i in range(len(coordinates) - 1):
        lat1, lon1 = coordinates[i]
        lat2, lon2 = coordinates[i + 1]
        
        segment_distance = haversine_distance(lat1, lon1, lat2, lon2)
        
        if segment_distance < spacing_m:
            interpolated_points.append(RoutePoint(
                index=point_index,
                lat=lat1,
                lon=lon1,
                distance_m=total_distance
            ))
            point_index += 1
        else:
            num_points = int(segment_distance / spacing_m)
            for j in range(num_points):
                fraction = j / num_points
                lat = lat1 + fraction * (lat2 - lat1)
                lon = lon1 + fraction * (lon2 - lon1)
                
                interpolated_points.append(RoutePoint(
                    index=point_index,
                    lat=lat,
                    lon=lon,
                    distance_m=total_distance + j * spacing_m
                ))
                point_index += 1
        
        total_distance += segment_distance
    
    # Add final point
    if coordinates:
        lat, lon = coordinates[-1]
        interpolated_points.append(RoutePoint(
            index=point_index,
            lat=lat,
            lon=lon,
            distance_m=total_distance
        ))
    
    print(f"  Created {len(interpolated_points)} interpolated points")
    print(f"  Total route distance: {total_distance/1000:.2f} km")
    
    # Now apply smart caching
    cache = SmartInterpolationCache(cache_dir)
    
    if kmz_path and os.path.exists(kmz_path):
        interpolated_points, index_mapping = cache.match_points_with_cache(
            interpolated_points, kmz_path
        )
        
        # Report on cache effectiveness
        points_with_elevation = sum(1 for p in interpolated_points if p.elevation is not None)
        if points_with_elevation > 0:
            print(f"\n  Cache effectiveness:")
            print(f"    Points with cached elevation: {points_with_elevation}/{len(interpolated_points)}")
            print(f"    Cache hit rate: {points_with_elevation/len(interpolated_points)*100:.1f}%")
            
            # Identify which points still need elevation extraction
            uncached_indices = [p.index for p in interpolated_points if p.elevation is None]
            if uncached_indices:
                print(f"    Points needing extraction: {len(uncached_indices)}")
                if len(uncached_indices) <= 10:
                    print(f"    Indices: {uncached_indices}")
                else:
                    print(f"    First 10 indices: {uncached_indices[:10]}")
    
    return interpolated_points

def calculate_route_bounds(points: List[RoutePoint], buffer_km: float = 2.0) -> Tuple[float, float, float, float]:
    """Calculate the bounding box of all route points with buffer"""
    lats = [p.lat for p in points]
    lons = [p.lon for p in points]
    
    buffer_deg_lat = buffer_km / 111.0
    buffer_deg_lon = buffer_km / (111.0 * math.cos(math.radians(np.mean(lats))))
    
    return (
        min(lons) - buffer_deg_lon,
        min(lats) - buffer_deg_lat,
        max(lons) + buffer_deg_lon,
        max(lats) + buffer_deg_lat
    )

# ==================== COORDINATE SYSTEM DETECTION ====================

def detect_lidar_crs(lidar_x: np.ndarray, lidar_y: np.ndarray, las_header=None, filename: str = None) -> Optional['CRS']:
    """
    Detect the coordinate reference system of LiDAR data.
    Works for all US states, handles both files with and without CRS metadata.
    
    Args:
        lidar_x: X coordinates from the LiDAR file
        lidar_y: Y coordinates from the LiDAR file
        las_header: Optional LAS file header containing VLR records
        filename: Optional filename for parsing state/zone info when headers are missing
    """
    if not PYPROJ_AVAILABLE:
        print("      ERROR: pyproj not available - cannot detect CRS")
        return None
    
    from pyproj import CRS, Transformer
    import re
    
    print(f"      Detecting CRS for LiDAR data...")
    
    # ========== STEP 1: Try to extract CRS from metadata ==========
    if las_header and hasattr(las_header, 'vlrs') and las_header.vlrs:
        print("      Checking LAS header for CRS information...")
        
        for vlr in las_header.vlrs:
            if not hasattr(vlr, 'record_id'):
                continue
                
            # WKT records (EPSG:2111 or 2112)
            if vlr.record_id in [2111, 2112]:
                try:
                    # Get WKT string
                    wkt_string = None
                    if hasattr(vlr, 'parse_crs'):
                        wkt_string = str(vlr.parse_crs())
                    elif hasattr(vlr, 'body'):
                        try:
                            wkt_string = vlr.body.decode('ascii', errors='ignore').strip('\x00')
                        except:
                            continue
                    
                    if wkt_string:
                        # Check if this is a compound CRS (COMPD_CS)
                        if 'COMPD_CS' in wkt_string:
                            print("      Found compound CRS in WKT")
                            
                            # Look for Ohio South specifically in the WKT
                            if 'Ohio South' in wkt_string:
                                print(f"      ✓ Detected Ohio South from compound CRS WKT")
                                return CRS.from_epsg(3735)  # Ohio South (ftUS)
                            elif 'Ohio North' in wkt_string:
                                print(f"      ✓ Detected Ohio North from compound CRS WKT")
                                return CRS.from_epsg(3734)  # Ohio North (ftUS)
                            
                            # Extract all EPSG codes from the WKT
                            epsg_matches = re.findall(r'AUTHORITY\["EPSG","(\d+)"\]', wkt_string)
                            
                            # Look for PROJCS within COMPD_CS which contains the horizontal CRS
                            projcs_match = re.search(r'PROJCS\[.*?AUTHORITY\["EPSG","(\d+)"\]\]', wkt_string, re.DOTALL)
                            
                            if projcs_match:
                                horizontal_epsg = projcs_match.group(1)
                                print(f"      ✓ Extracted horizontal CRS from compound: EPSG:{horizontal_epsg}")
                                
                                # Verify it's a known state plane zone
                                if horizontal_epsg == '6551':  # Ohio South compound
                                    print(f"      ✓ EPSG:6551 is Ohio South compound CRS")
                                    return CRS.from_epsg(3735)
                                elif horizontal_epsg == '6549':  # Ohio North compound
                                    print(f"      ✓ EPSG:6549 is Ohio North compound CRS")
                                    return CRS.from_epsg(3734)
                                
                                try:
                                    detected_crs = CRS.from_epsg(int(horizontal_epsg))
                                    crs_name = detected_crs.to_string()
                                    print(f"      ✓ Using horizontal CRS: {crs_name[:50]}...")
                                    return detected_crs
                                except:
                                    pass
                            
                            # Fallback: Map known compound/vertical CRS to horizontal component
                            compound_to_horizontal = {
                                '6549': 3734,  # NAD83(2011) / Ohio North (ftUS) + NAVD88 -> Ohio North
                                '6550': 3735,  # NAD83(2011) / Ohio South (ftUS) + NAVD88 -> Ohio South
                                '6551': 3735,  # NAD83(2011) / Ohio South (ftUS) -> Ohio South
                                '6360': 3735,  # NAVD88 height - Geoid18 (ftUS) in Ohio context -> Ohio South
                            }
                            
                            # Try each EPSG code found
                            for epsg_code in epsg_matches:
                                if epsg_code in compound_to_horizontal:
                                    horizontal_epsg = compound_to_horizontal[epsg_code]
                                    print(f"      ✓ Found compound CRS EPSG:{epsg_code} -> using EPSG:{horizontal_epsg}")
                                    return CRS.from_epsg(horizontal_epsg)
                        
                        # Not a compound CRS, try normal PROJCS extraction
                        elif 'PROJCS' in wkt_string:
                            # Extract EPSG codes from WKT
                            epsg_matches = re.findall(r'AUTHORITY\["EPSG","(\d+)"\]', wkt_string)
                            
                            if epsg_matches:
                                # Map compound CRS to horizontal component
                                compound_to_horizontal = {
                                    '6549': 3734,  # NAD83(2011) / Ohio North (ftUS) + NAVD88 -> Ohio North
                                    '6550': 3735,  # NAD83(2011) / Ohio South (ftUS) + NAVD88 -> Ohio South
                                    '6551': 3735,  # Ohio South compound
                                    '6360': 3735,  # When found alone, likely Ohio South context
                                }
                                
                                # Try the last EPSG (most specific)
                                epsg_code = epsg_matches[-1]
                                
                                if epsg_code in compound_to_horizontal:
                                    horizontal_epsg = compound_to_horizontal[epsg_code]
                                    print(f"      ✓ Found compound CRS EPSG:{epsg_code} -> using EPSG:{horizontal_epsg}")
                                    return CRS.from_epsg(horizontal_epsg)
                                else:
                                    try:
                                        detected_crs = CRS.from_epsg(int(epsg_code))
                                        print(f"      ✓ Found CRS in metadata: EPSG:{epsg_code}")
                                        return detected_crs
                                    except:
                                        pass
                        
                        # Try parsing WKT directly if EPSG extraction failed
                        try:
                            detected_crs = CRS.from_wkt(wkt_string)
                            if detected_crs:
                                epsg = detected_crs.to_epsg()
                                if epsg:
                                    print(f"      ✓ Parsed WKT to EPSG:{epsg}")
                                    return detected_crs
                        except:
                            pass
                            
                except Exception as e:
                    continue
    
    # ========== STEP 2: Try to parse filename for state/zone info ==========
    if filename:
        print(f"      Checking filename for state/zone information: {filename}")
        
        # Convert filename to uppercase for matching
        fname_upper = filename.upper()
        
        # State abbreviation to zone mappings (for common USGS naming patterns)
        state_zone_mappings = {
            # Ohio
            ('OH', 'NORTH'): 3734,  # Ohio North (feet)
            ('OH', 'SOUTH'): 3735,  # Ohio South (feet)
            ('OHIO', 'NORTH'): 3734,
            ('OHIO', 'SOUTH'): 3735,
            # Indiana
            ('IN', 'EAST'): 3070,
            ('IN', 'WEST'): 3071,
            ('INDIANA', 'EAST'): 3070,
            ('INDIANA', 'WEST'): 3071,
            # Kentucky
            ('KY', 'NORTH'): 2246,
            ('KY', 'SOUTH'): 2247,
            ('KENTUCKY', 'NORTH'): 2246,
            ('KENTUCKY', 'SOUTH'): 2247,
            # Michigan
            ('MI', 'NORTH'): 2251,
            ('MI', 'CENTRAL'): 2252,
            ('MI', 'SOUTH'): 2253,
            ('MICHIGAN', 'NORTH'): 2251,
            ('MICHIGAN', 'CENTRAL'): 2252,
            ('MICHIGAN', 'SOUTH'): 2253,
            # Pennsylvania
            ('PA', 'NORTH'): 2271,
            ('PA', 'SOUTH'): 2272,
            ('PENNSYLVANIA', 'NORTH'): 2271,
            ('PENNSYLVANIA', 'SOUTH'): 2272,
            # Illinois
            ('IL', 'EAST'): 3435,
            ('IL', 'WEST'): 3436,
            ('ILLINOIS', 'EAST'): 3435,
            ('ILLINOIS', 'WEST'): 3436,
            # New York
            ('NY', 'EAST'): 2260,
            ('NY', 'CENTRAL'): 2261,
            ('NY', 'WEST'): 2262,
            ('NY', 'LONG_ISLAND'): 2263,
            ('NEW_YORK', 'EAST'): 2260,
            ('NEW_YORK', 'CENTRAL'): 2261,
            ('NEW_YORK', 'WEST'): 2262,
            # Florida
            ('FL', 'EAST'): 2236,
            ('FL', 'WEST'): 2237,
            ('FL', 'NORTH'): 2238,
            ('FLORIDA', 'EAST'): 2236,
            ('FLORIDA', 'WEST'): 2237,
            ('FLORIDA', 'NORTH'): 2238,
            # Texas
            ('TX', 'NORTH'): 2275,
            ('TX', 'NORTH_CENTRAL'): 2276,
            ('TX', 'CENTRAL'): 2277,
            ('TX', 'SOUTH_CENTRAL'): 2278,
            ('TX', 'SOUTH'): 2279,
            ('TEXAS', 'NORTH'): 2275,
            ('TEXAS', 'NORTH_CENTRAL'): 2276,
            ('TEXAS', 'CENTRAL'): 2277,
            ('TEXAS', 'SOUTH_CENTRAL'): 2278,
            ('TEXAS', 'SOUTH'): 2279,
            # California
            ('CA', '1'): 2225, ('CA', '2'): 2226, ('CA', '3'): 2227,
            ('CA', '4'): 2228, ('CA', '5'): 2229, ('CA', '6'): 2230,
            ('CALIFORNIA', '1'): 2225, ('CALIFORNIA', '2'): 2226,
            # Add more states as needed
        }
        
        # Look for state and zone patterns in filename
        for (state, zone), epsg in state_zone_mappings.items():
            # Check for patterns like "OH_NORTH", "OH_South", "OHIO_NORTH", etc.
            if state in fname_upper and zone in fname_upper:
                # Verify they appear in reasonable proximity (within 20 characters)
                state_pos = fname_upper.find(state)
                zone_pos = fname_upper.find(zone, state_pos)
                if zone_pos != -1 and zone_pos - state_pos < 20:
                    print(f"      ✓ Detected from filename: {state} {zone} -> EPSG:{epsg}")
                    return CRS.from_epsg(epsg)
        
        # Alternative patterns for USGS naming (e.g., "USGS_LPC_OH_NORTH_2006")
        pattern = r'(?:USGS_)?(?:LPC_)?([A-Z]{2,})_([A-Z_]+)_(?:\d{4})'
        match = re.search(pattern, fname_upper)
        if match:
            state_str = match.group(1)
            zone_str = match.group(2)
            
            # Try to find matching EPSG
            for (state, zone), epsg in state_zone_mappings.items():
                if state_str == state and zone_str == zone:
                    print(f"      ✓ Detected from filename pattern: {state} {zone} -> EPSG:{epsg}")
                    return CRS.from_epsg(epsg)
    
    # ========== STEP 3: Test State Plane zones based on coordinates ==========
    x_center = (lidar_x.min() + lidar_x.max()) / 2
    y_center = (lidar_y.min() + lidar_y.max()) / 2
    
    print(f"      No CRS in metadata or filename, using coordinate-based detection")
    print(f"      Center point: X={x_center:.1f}, Y={y_center:.1f}")
    
    # Check if geographic coordinates
    if -180 <= x_center <= 180 and -90 <= y_center <= 90:
        print("      Detected: Geographic coordinates (WGS84)")
        return CRS.from_epsg(4326)
    
    # Determine likely units based on magnitude
    is_likely_feet = x_center > 300000 or y_center > 300000
    
    # Common US State Plane zones
    # Only include the most commonly used zones to speed up detection
    common_zones = [
        # Feet zones (most common for USGS data)
        (3734, 'Ohio North (feet)'),
        (3735, 'Ohio South (feet)'),
        (3070, 'Indiana East (feet)'),
        (3071, 'Indiana West (feet)'),
        (2246, 'Kentucky North (feet)'),
        (2247, 'Kentucky South (feet)'),
        (2251, 'Michigan North (feet)'),
        (2252, 'Michigan Central (feet)'),
        (2253, 'Michigan South (feet)'),
        (2271, 'Pennsylvania North (feet)'),
        (2272, 'Pennsylvania South (feet)'),
        (3435, 'Illinois East (feet)'),
        (3436, 'Illinois West (feet)'),
        (2260, 'New York East (feet)'),
        (2261, 'New York Central (feet)'),
        (2262, 'New York West (feet)'),
        (2263, 'New York Long Island (feet)'),
        (2236, 'Florida East (feet)'),
        (2237, 'Florida West (feet)'),
        (2238, 'Florida North (feet)'),
        (2275, 'Texas North (feet)'),
        (2276, 'Texas North Central (feet)'),
        (2277, 'Texas Central (feet)'),
        (2278, 'Texas South Central (feet)'),
        (2279, 'Texas South (feet)'),
        (2225, 'California 1 (feet)'),
        (2226, 'California 2 (feet)'),
        (2227, 'California 3 (feet)'),
        (2228, 'California 4 (feet)'),
        (2229, 'California 5 (feet)'),
        (2230, 'California 6 (feet)'),
        # Meter zones (less common but still used)
        (32122, 'Ohio South (meters)'),
        (32123, 'Ohio North (meters)'),
        (26973, 'Indiana East (meters)'),
        (26974, 'Indiana West (meters)'),
        (26979, 'Kentucky North (meters)'),
        (26980, 'Kentucky South (meters)'),
        # Add more zones as needed
    ]
    
    print(f"      Testing common State Plane zones...")
    
    # Test zones
    for epsg_code, zone_name in common_zones:
        # Skip meter zones if coordinates look like feet
        if is_likely_feet and 'meters' in zone_name:
            continue
        # Skip feet zones if coordinates look like meters  
        if not is_likely_feet and 'feet' in zone_name:
            continue
            
        try:
            test_crs = CRS.from_epsg(epsg_code)
            transformer = Transformer.from_crs(test_crs, CRS.from_epsg(4326), always_xy=True)
            lon, lat = transformer.transform(x_center, y_center)
            
            # Check if result is in valid US bounds
            if (24 < lat < 50) and (-125 < lon < -65):
                print(f"      ✓ DETECTED: {zone_name} (EPSG:{epsg_code})")
                print(f"        Location: {lat:.4f}°N, {lon:.4f}°W")
                return test_crs
                
        except Exception:
            continue
    
    # If no match found in common zones, test remaining zones
    if is_likely_feet:
        print("      No match in common zones, testing all US feet-based zones...")
        # Extended list of all US feet zones
        all_feet_zones = [
            (102629, 'Alabama East'), (102630, 'Alabama West'),
            (2222, 'Arizona East'), (2223, 'Arizona Central'), (2224, 'Arizona West'),
            (3433, 'Arkansas North'), (3434, 'Arkansas South'),
            (2231, 'Colorado North'), (2232, 'Colorado Central'), (2233, 'Colorado South'),
            (2234, 'Connecticut'),
            (2235, 'Delaware'),
            (4019, 'Georgia East'), (4020, 'Georgia West'),
            (2241, 'Idaho East'), (2242, 'Idaho Central'), (2243, 'Idaho West'),
            (3417, 'Iowa North'), (3418, 'Iowa South'),
            (3419, 'Kansas North'), (3420, 'Kansas South'),
            (3089, 'Kentucky Single Zone'),
            (3451, 'Louisiana North'), (3452, 'Louisiana South'),
            (2249, 'Maine East'), (2250, 'Maine West'),
            (2248, 'Maryland'),
            (3594, 'Minnesota North'), (3595, 'Minnesota Central'), (3596, 'Minnesota South'),
            (2254, 'Mississippi East'), (2255, 'Mississippi West'),
            (2256, 'Montana'),
            (2257, 'Nebraska'),
            (3421, 'Nevada East'), (3422, 'Nevada Central'), (3423, 'Nevada West'),
            (2258, 'New Hampshire'),
            (2260, 'New Jersey'),
            (2264, 'North Carolina'),
            (2265, 'North Dakota North'), (2266, 'North Dakota South'),
            (2267, 'Oklahoma North'), (2268, 'Oklahoma South'),
            (2269, 'Oregon North'), (2270, 'Oregon South'),
            (3438, 'Rhode Island'),
            (3900, 'South Carolina'),
            (3454, 'South Dakota North'), (3455, 'South Dakota South'),
            (2274, 'Tennessee'),
            (3566, 'Utah North'), (3567, 'Utah Central'), (3568, 'Utah South'),
            (5646, 'Vermont'),
            (2283, 'Virginia North'), (2284, 'Virginia South'),
            (2285, 'Washington North'), (2286, 'Washington South'),
            (2287, 'Wisconsin North'), (2288, 'Wisconsin Central'), (2289, 'Wisconsin South'),
        ]
        
        for epsg_code, zone_name in all_feet_zones:
            try:
                test_crs = CRS.from_epsg(epsg_code)
                transformer = Transformer.from_crs(test_crs, CRS.from_epsg(4326), always_xy=True)
                lon, lat = transformer.transform(x_center, y_center)
                
                if (24 < lat < 50) and (-125 < lon < -65):
                    print(f"      ✓ DETECTED: {zone_name} (feet) (EPSG:{epsg_code})")
                    print(f"        Location: {lat:.4f}°N, {lon:.4f}°W")
                    return test_crs
            except:
                continue
    
    print("      WARNING: Could not determine State Plane zone")
    return None

# Additional helper function to add to the notebook (optional but recommended)
def print_lidar_diagnostic_info(las_file_path):
    """
    Enhanced diagnostic function for LiDAR files using stateplane package.
    """
    if not LASPY_AVAILABLE:
        print("Error: laspy not available for diagnosis")
        return
    
    try:
        import laspy
        with laspy.open(las_file_path) as f:
            las = f.read()
            
            print(f"\n{'='*60}")
            print(f"LiDAR File Diagnostic: {las_file_path.name}")
            print(f"{'='*60}")
            
            print(f"\nPoint Statistics:")
            print(f"  Count: {len(las.x):,}")
            print(f"  X range: [{las.x.min():.2f}, {las.x.max():.2f}] (span: {las.x.max()-las.x.min():.2f})")
            print(f"  Y range: [{las.y.min():.2f}, {las.y.max():.2f}] (span: {las.y.max()-las.y.min():.2f})")
            print(f"  Z range: [{las.z.min():.2f}, {las.z.max():.2f}] (span: {las.z.max()-las.z.min():.2f})")
            
            print(f"\nCoordinate System Detection:")
            detected_crs = detect_lidar_crs(las.x, las.y, las.header)
            
            if detected_crs:
                epsg = detected_crs.to_epsg()
                print(f"\n  Detected CRS: {detected_crs.to_string()[:100]}...")
                if epsg:
                    print(f"  EPSG Code: {epsg}")
                
                # If stateplane is available, show verification
                if STATEPLANE_AVAILABLE and epsg:
                    try:
                        from pyproj import Transformer
                        transformer = Transformer.from_crs(
                            detected_crs,
                            CRS.from_epsg(4326),
                            always_xy=True
                        )
                        x_center = (las.x.min() + las.x.max()) / 2
                        y_center = (las.y.min() + las.y.max()) / 2
                        lon, lat = transformer.transform(x_center, y_center)
                        
                        print(f"\n  Geographic location (center):")
                        print(f"    Latitude: {lat:.6f}°")
                        print(f"    Longitude: {lon:.6f}°")
                        
                        import stateplane
                        identified_fips = stateplane.identify(lon, lat, fmt='fips')
                        identified_epsg = stateplane.identify(lon, lat, fmt='epsg')
                        
                        if identified_fips:
                            print(f"\n  Stateplane verification:")
                            print(f"    FIPS code: {identified_fips}")
                            print(f"    Suggested EPSG: {identified_epsg}")
                    except Exception as e:
                        pass
            
            if hasattr(las.header, 'vlrs') and las.header.vlrs:
                print(f"\n  VLR Records: {len(las.header.vlrs)} found")
                for i, vlr in enumerate(las.header.vlrs):
                    if hasattr(vlr, 'record_id'):
                        if vlr.record_id in [2111, 2112, 34735, 34737]:
                            print(f"    VLR {i}: Record ID {vlr.record_id} (CRS information)")
            
            print(f"{'='*60}\n")
            
    except Exception as e:
        print(f"Error reading file: {e}")

# ==================== LIDAR INTERPOLATION CHECKING ====================

def point_can_be_interpolated_from_tile(point: RoutePoint, tile: ElevationTile) -> bool:
    """Check if a point can be interpolated from a LiDAR tile"""
    if not LIDAR_INTERPOLATION_ENABLED:
        return False
    
    min_lon, min_lat, max_lon, max_lat = tile.bbox
    
    lat_buffer = LIDAR_INTERPOLATION_RADIUS_M / 111000.0
    lon_buffer = LIDAR_INTERPOLATION_RADIUS_M / (111000.0 * math.cos(math.radians(point.lat)))
    
    extended_min_lon = min_lon - lon_buffer
    extended_min_lat = min_lat - lat_buffer
    extended_max_lon = max_lon + lon_buffer
    extended_max_lat = max_lat + lat_buffer
    
    return (extended_min_lon <= point.lon <= extended_max_lon and 
            extended_min_lat <= point.lat <= extended_max_lat)

def check_lidar_interpolation_needed(point: RoutePoint, tile: ElevationTile) -> bool:
    """Check if a point within a LiDAR tile needs interpolation"""
    if not LIDAR_INTERPOLATION_ENABLED:
        return False
    
    tile_center_lat = (tile.bbox[1] + tile.bbox[3]) / 2
    meters_per_deg_lat = 111000
    meters_per_deg_lon = 111000 * math.cos(math.radians(tile_center_lat))
    
    point_x_m = (point.lon - tile.bbox[0]) * meters_per_deg_lon
    point_y_m = (point.lat - tile.bbox[1]) * meters_per_deg_lat
    
    x_offset = point_x_m % LIDAR_POINT_SPACING_M
    y_offset = point_y_m % LIDAR_POINT_SPACING_M
    
    threshold = LIDAR_POINT_SPACING_M * 0.1
    needs_interpolation = (
        min(x_offset, LIDAR_POINT_SPACING_M - x_offset) > threshold or
        min(y_offset, LIDAR_POINT_SPACING_M - y_offset) > threshold
    )
    
    return needs_interpolation

# ==================== POINT-BASED TILE SEARCHER V2 ====================

class PointBasedElevationSearcher:
    """Search for elevation tiles with early termination when coverage is complete"""
    
    def __init__(self, cache_dir: Path = CACHE_DIR):
        self.session = requests.Session()
        self.session.headers.update({'User-Agent': 'Python Elevation Searcher 1.0'})
        self.cache_dir = cache_dir
        self.cache_file = self.cache_dir / "tile_cache.json"
        self.tile_cache = self._load_cache()
        self.api_call_count = 0
        self.cache_hits = 0
        
        self.data_sources = [
            {
                'priority': 1,
                'data_type': 'lidar',
                'datasets': [
                    'Lidar Point Cloud (LPC)', 
                    'USGS Lidar Point Cloud',
                    'USGS 3DEP LPC',
                    '3DEP LiDAR Point Cloud',
                    'Lidar Point Cloud',
                    'USGS LPC Current',
                    'LPC',
                    '3DEP Point Cloud'
                ],
                'formats': ['LAZ', 'LAS', 'laz', 'las'],  # Include lowercase
                'description': 'LiDAR Point Clouds',
                'color': 'purple'
            },
            {
                'priority': 2,
                'data_type': 'dem_sub1m',
                'datasets': ['Digital Elevation Model (DEM) 50 centimeter'],
                'formats': ['GeoTIFF'],
                'description': 'Sub-meter DEMs',
                'color': 'blue'
            },
            {
                'priority': 3,
                'data_type': 'dem_1m',
                'datasets': [
                    'Digital Elevation Model (DEM) 1 meter',
                    '3DEP 1 Meter',
                    'USGS 1 Meter DEM',
                    'National Elevation Dataset (NED) 1 meter'
                ],
                'formats': ['GeoTIFF', 'IMG'],
                'description': '1-meter DEMs',
                'color': 'green'
            },
            {
                'priority': 4,
                'data_type': 'dem_10m',
                'datasets': [
                    'National Elevation Dataset (NED) 1/3 arc-second',
                    'USGS NED 10 Meter',
                    '3DEP 10 Meter'
                ],
                'formats': ['GeoTIFF', 'IMG'],
                'description': '10-meter DEMs',
                'color': 'yellow'
            }
        ]
    
    def _load_cache(self) -> Dict:
        """Load tile cache from disk"""
        if self.cache_file.exists():
            try:
                with open(self.cache_file, 'r') as f:
                    cache = json.load(f)
                    if 'tiles' not in cache:
                        cache['tiles'] = {}
                    if isinstance(cache['tiles'], list):
                        cache['tiles'] = {}
                    print(f"   Loaded cache with {len(cache.get('tiles', {}))} cached tile records")
                    return cache
            except Exception as e:
                print(f"   Warning: Could not load cache: {e}")
        return {'tiles': {}, 'last_updated': time.time()}
    
    def _save_cache(self):
        """Save tile cache to disk"""
        try:
            if 'tiles' not in self.tile_cache:
                self.tile_cache['tiles'] = {}
            elif not isinstance(self.tile_cache['tiles'], dict):
                self.tile_cache['tiles'] = {}
            
            self.tile_cache['last_updated'] = time.time()
            with open(self.cache_file, 'w') as f:
                json.dump(self.tile_cache, f, indent=2)
        except Exception as e:
            print(f"   Warning: Could not save cache: {e}")
    
    def _get_cache_key(self, bbox: Tuple[float, float, float, float], source: Dict) -> str:
        """Generate cache key for a search"""
        min_lon, min_lat, max_lon, max_lat = bbox
        return f"{source['data_type']}_{min_lon:.4f}_{min_lat:.4f}_{max_lon:.4f}_{max_lat:.4f}"
    
    def search_tiles_for_points(self, points: List[RoutePoint], 
                               full_points_list: List[RoutePoint] = None) -> Dict[str, List[ElevationTile]]:
        """Search for tiles with early termination
        
        Args:
            points: Filtered list of points that need elevation data
            full_points_list: Original complete list of all route points (for index lookups)
        """
        # If full list not provided, assume points is the full list
        if full_points_list is None:
            full_points_list = points
            
        print("\n" + "="*60)
        print("POINT-BASED TILE SEARCH WITH EARLY INTERPOLATION CHECK")
        print("="*60)
        
        # Print search configuration
        print("\nSearch Configuration:")
        print(f"  LiDAR overlap search: {LIDAR_SEARCH_OVERLAP}")
        print(f"  LiDAR search buffer: {LIDAR_SEARCH_BUFFER_DEG} degrees")
        print(f"  Force LiDAR re-search: {FORCE_LIDAR_RESEARCH}")
        if MANUAL_LIDAR_TILES:
            print(f"  Manual LiDAR tiles: {len(MANUAL_LIDAR_TILES)} specified")
        
        tiles_by_type = {}
        all_unique_tiles = {}
        uncovered_point_indices = set(range(len(points)))
        
        lidar_source = self.data_sources[0]
        
        print(f"\n1. Searching for LiDAR Point Clouds...")
        print(f"   Dataset names: {', '.join(lidar_source['datasets'][:3])}...")
        print(f"   Formats: {', '.join(lidar_source['formats'])}")
        print(f"   Points still needing coverage: {len(uncovered_point_indices)}")
        
        lidar_tiles = self._search_lidar_and_check_interpolation(
            points,
            full_points_list,  # Pass the full list too
            uncovered_point_indices,
            all_unique_tiles
        )
        
        if lidar_tiles:
            tiles_by_type['lidar'] = lidar_tiles
            print(f"   ✓ Found {len(lidar_tiles)} LiDAR tiles")
            
            # Print tile names for debugging
            if len(lidar_tiles) <= 10:
                for tile in lidar_tiles[:5]:
                    print(f"      - {tile.name[:60]}")
                if len(lidar_tiles) > 5:
                    print(f"      ... and {len(lidar_tiles) - 5} more")
            
            # Mark all covered points (both direct and interpolatable)
            # The interpolation checking already happened in _search_lidar_and_check_interpolation
            for tile in lidar_tiles:
                for point_idx in tile.covered_points:
                    if point_idx in uncovered_point_indices and point_idx < len(full_points_list):
                        full_points_list[point_idx].data_source = 'lidar'
                        full_points_list[point_idx].source_priority = 1
                        # Check if this point was marked for interpolation
                        if point_idx in tile.interpolated_points:
                            full_points_list[point_idx].is_interpolated = True
                        uncovered_point_indices.remove(point_idx)
            
            covered_count = len([p for p in points if p.data_source == 'lidar'])
            direct_count = len([p for p in points if p.data_source == 'lidar' and not p.is_interpolated])
            interp_count = len([p for p in points if p.data_source == 'lidar' and p.is_interpolated])
            
            print(f"   Total LiDAR coverage: {covered_count} points ({direct_count} direct, {interp_count} interpolated)")
            
            if not uncovered_point_indices:
                print(f"   ✓ COMPLETE COVERAGE ACHIEVED with LiDAR")
                print(f"\n   Skipping all lower-priority data sources")
                for i, source in enumerate(self.data_sources[1:], start=2):
                    print(f"\n{i}. Skipping {source['description']} - all points already covered")
        else:
            print(f"   ✗ No LiDAR tiles found")
            print(f"   Consider:")
            print(f"     1. Setting FORCE_LIDAR_RESEARCH = True to bypass cache")
            print(f"     2. Setting LIDAR_SEARCH_OVERLAP = True for better coverage")
            print(f"     3. Adding known tile URLs to MANUAL_LIDAR_TILES list")
        
        if uncovered_point_indices:
            for source in self.data_sources[1:]:
                if not uncovered_point_indices:
                    print(f"\n{source['priority']}. Skipping {source['description']} - all points already covered")
                    continue
                
                print(f"\n{source['priority']}. Searching for {source['description']}...")
                print(f"   Points still needing coverage: {len(uncovered_point_indices)}")
                
                source_tiles = []
                uncovered_points = [points[i] for i in uncovered_point_indices]
                point_containing_tiles = self._find_tiles_containing_points(uncovered_points, source)
                
                if point_containing_tiles:
                    print(f"   Found {len(point_containing_tiles)} tiles containing route points")
                    
                    for tile in point_containing_tiles.values():
                        if tile.url not in all_unique_tiles:
                            source_tiles.append(tile)
                            all_unique_tiles[tile.url] = tile
                    
                    if source_tiles:
                        tiles_by_type[source['data_type']] = source_tiles
                        print(f"   ✓ Total {len(source_tiles)} unique tiles to cache")
                        
                        points_covered_count = 0
                        for tile in source_tiles:
                            for point_idx in tile.covered_points:
                                if points[point_idx].data_source is None or points[point_idx].source_priority > source['priority']:
                                    points[point_idx].data_source = source['data_type']
                                    points[point_idx].source_priority = source['priority']
                                    if point_idx in uncovered_point_indices:
                                        uncovered_point_indices.remove(point_idx)
                                        points_covered_count += 1
                        
                        print(f"   Covered {points_covered_count} additional points")
                        
                        if not uncovered_point_indices:
                            print(f"   ✓ COMPLETE COVERAGE ACHIEVED with {source['description']}")
                            break
                else:
                    print(f"   ✗ No tiles found")
        
        self._save_cache()
        print(f"\n   Final stats: {self.api_call_count} API calls, {self.cache_hits} cache hits")
        
        uncovered_points = [p for p in points if p.data_source is None]
        if uncovered_points:
            print(f"\n5. Using OpenTopography for {len(uncovered_points)} uncovered points...")
            opentopo_tile = self._create_opentopo_coverage(uncovered_points)
            tiles_by_type['opentopo'] = [opentopo_tile]
            for p in uncovered_points:
                p.data_source = 'opentopo'
                p.source_priority = 5
        
        return tiles_by_type
    
    def _search_lidar_and_check_interpolation(self, points: List[RoutePoint],
                                             full_points_list: List[RoutePoint],
                                             uncovered_indices: Set[int],
                                             all_unique_tiles: Dict) -> List[ElevationTile]:
        """Special handling for LiDAR search with interpolation checking
        
        Args:
            points: Filtered list of points needing elevation
            full_points_list: Complete original list for index-based access
            uncovered_indices: Set of indices that need coverage
            all_unique_tiles: Dictionary of already found tiles
        """
        lidar_source = self.data_sources[0]
        lidar_tiles = []
        
        # Use full_points_list for index access
        uncovered_points = [full_points_list[i] for i in uncovered_indices if i < len(full_points_list)]
        
        # First, do the standard search
        point_containing_tiles = self._find_tiles_containing_points(uncovered_points, lidar_source)
        
        # Check coverage including potential interpolation
        directly_covered_indices = set()
        interpolatable_indices = set()
        
        if point_containing_tiles:
            # Check direct coverage
            for tile in point_containing_tiles.values():
                directly_covered_indices.update(tile.covered_points)
            
            # Check interpolation potential if enabled
            if LIDAR_INTERPOLATION_ENABLED:
                for p in uncovered_points:
                    if p.index not in directly_covered_indices:
                        for tile in point_containing_tiles.values():
                            if point_can_be_interpolated_from_tile(p, tile):
                                interpolatable_indices.add(p.index)
                                break
            
            total_potential_coverage = len(directly_covered_indices) + len(interpolatable_indices)
            print(f"   Standard search: {len(directly_covered_indices)} direct, {len(interpolatable_indices)} interpolatable")
        else:
            total_potential_coverage = 0
        
        # Calculate remaining uncovered points after standard search + interpolation
        all_covered = directly_covered_indices.union(interpolatable_indices)
        remaining_uncovered = set(p.index for p in uncovered_points) - all_covered
        
        # Only do overlapping search if we still have uncovered points and it's enabled
        if LIDAR_SEARCH_OVERLAP and remaining_uncovered and len(remaining_uncovered) > 0:
            print(f"   {len(remaining_uncovered)} points still uncovered after standard search + interpolation")
            print(f"   Performing overlapping LiDAR searches for better coverage...")
            
            # Get the points that are still uncovered
            still_uncovered_points = [p for p in uncovered_points if p.index in remaining_uncovered]
            
            # Search with smaller groups for more granular coverage
            small_group_tiles = self._find_tiles_containing_points_with_overlap(
                still_uncovered_points, lidar_source, max_group_size=20
            )
            
            # Add new tiles to our collection
            new_tiles_count = 0
            for url, tile in small_group_tiles.items():
                if url not in point_containing_tiles:
                    point_containing_tiles[url] = tile
                    new_tiles_count += 1
                else:
                    # Merge covered points if tile already exists
                    point_containing_tiles[url].covered_points.update(tile.covered_points)
            
            # Also do a single large bbox search for the entire route
            route_bbox = self._get_route_bbox(still_uncovered_points)
            route_tiles = self._search_tiles_in_bbox(route_bbox, lidar_source)
            
            # Check which points each tile contains
            for tile in route_tiles:
                contained_points = self._get_contained_points(tile, uncovered_points)
                if contained_points and tile.url not in point_containing_tiles:
                    tile.covered_points = contained_points
                    point_containing_tiles[tile.url] = tile
                    new_tiles_count += 1
            
            # Re-check coverage after overlapping search
            new_direct = set()
            new_interpolatable = set()
            
            for tile in point_containing_tiles.values():
                new_direct.update(tile.covered_points)
            
            if LIDAR_INTERPOLATION_ENABLED:
                for p in uncovered_points:
                    if p.index not in new_direct:
                        for tile in point_containing_tiles.values():
                            if point_can_be_interpolated_from_tile(p, tile):
                                new_interpolatable.add(p.index)
                                break
            
            additional_covered = (len(new_direct) + len(new_interpolatable)) - total_potential_coverage
            
            print(f"   Overlapping search added {new_tiles_count} new tiles")
            print(f"   Additional coverage: {additional_covered} points")
            print(f"   Total tiles: {len(point_containing_tiles)}")
        elif point_containing_tiles:
            if len(remaining_uncovered) == 0:
                print(f"   Standard search + interpolation provides full coverage")
            else:
                print(f"   Overlapping search disabled, {len(remaining_uncovered)} points remain uncovered")
        
        # Add manual LiDAR tiles if specified
        if MANUAL_LIDAR_TILES:
            print(f"   Adding {len(MANUAL_LIDAR_TILES)} manual LiDAR tiles...")
            for url in MANUAL_LIDAR_TILES:
                if url and url not in point_containing_tiles:
                    # Create a tile for this manual URL
                    # Extract name from URL
                    name = url.split('/')[-1] if '/' in url else url
                    
                    # Create a bounding box that covers all points (will be checked during extraction)
                    route_bbox = self._get_route_bbox(uncovered_points) if uncovered_points else (0, 0, 0, 0)
                    
                    tile = ElevationTile(
                        name=f"Manual: {name[:50]}",
                        bbox=route_bbox,
                        url=url,
                        resolution=0.5,  # Assume high resolution for LiDAR
                        data_type='lidar',
                        priority=1,
                        format='LAZ' if url.lower().endswith('.laz') else 'LAS'
                    )
                    
                    # Assume it could cover all uncovered points (will be verified during extraction)
                    tile.covered_points = set(p.index for p in uncovered_points)
                    point_containing_tiles[url] = tile
                    print(f"     Added manual tile: {name}")
        
        # Now process all tiles and mark interpolation needs
        if point_containing_tiles:
            for tile in point_containing_tiles.values():
                if tile.url not in all_unique_tiles:
                    # Mark points that need interpolation
                    for point_idx in list(tile.covered_points):
                        if point_idx < len(full_points_list) and check_lidar_interpolation_needed(full_points_list[point_idx], tile):
                            tile.interpolated_points.add(point_idx)
                            full_points_list[point_idx].is_interpolated = True
                    
                    # Also add interpolatable points that weren't directly covered
                    if LIDAR_INTERPOLATION_ENABLED:
                        for p in uncovered_points:
                            if p.index not in tile.covered_points:
                                if point_can_be_interpolated_from_tile(p, tile):
                                    tile.covered_points.add(p.index)
                                    tile.interpolated_points.add(p.index)
                                    # Use full list for index access
                                    if p.index < len(full_points_list):
                                        full_points_list[p.index].is_interpolated = True
                    
                    lidar_tiles.append(tile)
                    all_unique_tiles[tile.url] = tile
            
            total_direct = sum(len(t.covered_points - t.interpolated_points) for t in lidar_tiles)
            total_interp = sum(len(t.interpolated_points) for t in lidar_tiles)
            if total_direct > 0 or total_interp > 0:
                print(f"   Final LiDAR coverage: {total_direct} direct, {total_interp} interpolated")
        
        return lidar_tiles
    
    def _find_tiles_containing_points_with_overlap(self, points: List[RoutePoint], 
                                                   source: Dict, 
                                                   max_group_size: int = 20) -> Dict[str, ElevationTile]:
        """Find tiles with overlapping search areas for better coverage"""
        tiles_found = {}
        
        # Create overlapping groups
        step = max(1, max_group_size // 2)  # 50% overlap
        for start_idx in range(0, len(points), step):
            end_idx = min(start_idx + max_group_size, len(points))
            group = points[start_idx:end_idx]
            
            if not group:
                continue
                
            group_bbox = self._get_group_bbox(group)
            tiles = self._search_tiles_in_bbox(group_bbox, source)
            
            for tile in tiles:
                if tile.url not in tiles_found:
                    contained_points = self._get_contained_points(tile, points)
                    if contained_points:
                        tile.covered_points = contained_points
                        tiles_found[tile.url] = tile
        
        return tiles_found
    
    def _get_route_bbox(self, points: List[RoutePoint]) -> Tuple[float, float, float, float]:
        """Get bounding box for entire route with buffer"""
        if not points:
            return (0, 0, 0, 0)
            
        lats = [p.lat for p in points]
        lons = [p.lon for p in points]
        
        # Use a larger buffer for route-level search
        buffer = LIDAR_SEARCH_BUFFER_DEG * 2
        
        return (
            min(lons) - buffer,
            min(lats) - buffer,
            max(lons) + buffer,
            max(lats) + buffer
        )
    
    def _find_tiles_containing_points(self, points: List[RoutePoint], source: Dict) -> Dict[str, ElevationTile]:
        """Find tiles that contain route points"""
        tiles_found = {}
        
        point_groups = self._group_points_by_proximity(points, max_group_size=50)
        print(f"   Searching {len(point_groups)} point groups...")
        
        for i, group in enumerate(point_groups):
            if i % 10 == 0 and i > 0:
                print(f"     Processed {i}/{len(point_groups)} groups...")
            
            group_bbox = self._get_group_bbox(group)
            tiles = self._search_tiles_in_bbox(group_bbox, source)
            
            for tile in tiles:
                if tile.url not in tiles_found:
                    contained_points = self._get_contained_points(tile, group)
                    
                    if contained_points:
                        tile.covered_points = contained_points
                        tiles_found[tile.url] = tile
        
        return tiles_found
    
    def _group_points_by_proximity(self, points: List[RoutePoint], max_group_size: int = 50) -> List[List[RoutePoint]]:
        """Group nearby points together for more efficient searching"""
        groups = []
        current_group = []
        
        for point in points:
            current_group.append(point)
            
            if len(current_group) >= max_group_size:
                groups.append(current_group)
                current_group = []
        
        if current_group:
            groups.append(current_group)
        
        return groups
    
    def _get_group_bbox(self, group: List[RoutePoint]) -> Tuple[float, float, float, float]:
        """Get bounding box for a group of points with small buffer"""
        lats = [p.lat for p in group]
        lons = [p.lon for p in group]
        
        # Use larger buffer for LiDAR searches if configured
        buffer = LIDAR_SEARCH_BUFFER_DEG if LIDAR_SEARCH_OVERLAP else 0.001
        
        return (
            min(lons) - buffer,
            min(lats) - buffer,
            max(lons) + buffer,
            max(lats) + buffer
        )
    
    def _search_tiles_in_bbox(self, bbox: Tuple[float, float, float, float], source: Dict) -> List[ElevationTile]:
        """Search for tiles in a specific bounding box with caching"""
        cache_key = self._get_cache_key(bbox, source)
        
        if 'tiles' not in self.tile_cache:
            self.tile_cache['tiles'] = {}
        elif not isinstance(self.tile_cache['tiles'], dict):
            self.tile_cache['tiles'] = {}
        
        # Check cache unless force re-search is enabled for LiDAR
        use_cache = not (FORCE_LIDAR_RESEARCH and source['data_type'] == 'lidar')
        
        if use_cache and cache_key in self.tile_cache['tiles']:
            self.cache_hits += 1
            cached_data = self.tile_cache['tiles'][cache_key]
            tiles = []
            for tile_data in cached_data:
                tile = ElevationTile(
                    name=tile_data['name'],
                    bbox=tuple(tile_data['bbox']),
                    url=tile_data['url'],
                    resolution=tile_data['resolution'],
                    data_type=tile_data['data_type'],
                    priority=tile_data['priority'],
                    format=tile_data.get('format')
                )
                tiles.append(tile)
            return tiles
        
        tiles = []
        min_lon, min_lat, max_lon, max_lat = bbox
        bbox_str = f"{min_lon},{min_lat},{max_lon},{max_lat}"
        
        # For LiDAR, try multiple search strategies
        if source['data_type'] == 'lidar':
            # Try each dataset name individually for better results
            for dataset in source['datasets']:
                for format_type in source['formats']:
                    params = {
                        'bbox': bbox_str,
                        'datasets': dataset,
                        'prodFormats': format_type,
                        'max': 100
                    }
                    
                    try:
                        self.api_call_count += 1
                        if self.api_call_count % 10 == 0:
                            print(f"     Made {self.api_call_count} API calls, {self.cache_hits} cache hits...")
                        
                        response = self.session.get(USGS_TNM_API, params=params, timeout=10)
                        response.raise_for_status()
                        data = response.json()
                        
                        if 'items' in data:
                            for item in data['items']:
                                tile = self._parse_tile(item, source)
                                if tile and tile.url not in [t.url for t in tiles]:
                                    tiles.append(tile)
                        
                        time.sleep(0.1)
                        
                    except Exception as e:
                        continue
            
            # Also try a wildcard search for LiDAR
            try:
                params = {
                    'bbox': bbox_str,
                    'prodFormats': 'LAZ,LAS',
                    'max': 100,
                    'q': 'lidar'  # Text search for "lidar"
                }
                
                response = self.session.get(USGS_TNM_API, params=params, timeout=10)
                response.raise_for_status()
                data = response.json()
                
                if 'items' in data:
                    for item in data['items']:
                        # Check if it's actually LiDAR data
                        title = item.get('title', '').lower()
                        format_type = item.get('format', '').upper()
                        if ('lidar' in title or 'lpc' in title or 'point cloud' in title) and \
                           (format_type in ['LAZ', 'LAS']):
                            tile = self._parse_tile(item, source)
                            if tile and tile.url not in [t.url for t in tiles]:
                                tiles.append(tile)
                
            except:
                pass
        else:
            # Original search for non-LiDAR sources
            for dataset in source['datasets']:
                for format_type in source['formats']:
                    params = {
                        'bbox': bbox_str,
                        'datasets': dataset,
                        'prodFormats': format_type,
                        'max': 100
                    }
                    
                    try:
                        self.api_call_count += 1
                        if self.api_call_count % 10 == 0:
                            print(f"     Made {self.api_call_count} API calls, {self.cache_hits} cache hits...")
                        
                        response = self.session.get(USGS_TNM_API, params=params, timeout=10)
                        response.raise_for_status()
                        data = response.json()
                        
                        if 'items' in data:
                            for item in data['items']:
                                tile = self._parse_tile(item, source)
                                if tile:
                                    tiles.append(tile)
                        
                        time.sleep(0.1)
                        
                    except Exception as e:
                        if self.api_call_count > 1:
                            print(f"     API error: {str(e)[:50]}")
                        continue
        
        cached_tiles = []
        for tile in tiles:
            cached_tiles.append({
                'name': tile.name,
                'bbox': list(tile.bbox),
                'url': tile.url,
                'resolution': tile.resolution,
                'data_type': tile.data_type,
                'priority': tile.priority,
                'format': tile.format
            })
        
        self.tile_cache['tiles'][cache_key] = cached_tiles
        
        if self.api_call_count % 20 == 0:
            self._save_cache()
        
        return tiles
    
    def _parse_tile(self, item: Dict, source: Dict) -> Optional[ElevationTile]:
        """Parse tile information from API response"""
        try:
            bbox = item.get('boundingBox', {})
            if not bbox:
                return None
            
            return ElevationTile(
                name=item.get('title', 'Unknown')[:100],
                bbox=(
                    float(bbox.get('minX', 0)),
                    float(bbox.get('minY', 0)),
                    float(bbox.get('maxX', 0)),
                    float(bbox.get('maxY', 0))
                ),
                url=item.get('downloadURL', ''),
                resolution=self._get_resolution(item, source),
                data_type=source['data_type'],
                priority=source['priority'],
                format=item.get('format', 'Unknown')
            )
        except Exception:
            return None
    
    def _get_resolution(self, item: Dict, source: Dict) -> float:
        """Extract resolution from tile metadata"""
        title = str(item.get('title', ''))
        
        if '50 centimeter' in title or '0.5m' in title:
            return 0.5
        elif '1 meter' in title or '1m' in title:
            return 1.0
        elif '10 meter' in title or '1/3 arc' in title:
            return 10.0
        elif 'lidar' in source['data_type']:
            return 0.5
        else:
            return 10.0
    
    def _get_contained_points(self, tile: ElevationTile, points: List[RoutePoint]) -> Set[int]:
        """Get indices of points contained within tile bounds"""
        contained = set()
        min_lon, min_lat, max_lon, max_lat = tile.bbox
        
        for point in points:
            if min_lon <= point.lon <= max_lon and min_lat <= point.lat <= max_lat:
                contained.add(point.index)
        
        return contained
    
    def _create_opentopo_coverage(self, points: List[RoutePoint]) -> ElevationTile:
        """Create virtual OpenTopography tile for uncovered points"""
        lats = [p.lat for p in points]
        lons = [p.lon for p in points]
        
        tile = ElevationTile(
            name="OpenTopography SRTM Coverage",
            bbox=(min(lons), min(lats), max(lons), max(lats)),
            url=None,
            resolution=30.0,
            data_type='opentopo',
            priority=5
        )
        
        tile.covered_points = set(p.index for p in points)
        return tile

    def merge_override_cache(self):
        """Merge temporary cache into main cache, overwriting conflicts"""
        if not self.override_cache or not self.temp_elevation_cache:
            return
        
        print("\n" + "="*60)
        print("MERGING OVERRIDE CACHE")
        print("="*60)
        
        if 'tiles' not in self.elevation_cache:
            self.elevation_cache['tiles'] = {}
        
        overwritten_count = 0
        new_count = 0
        total_points_updated = 0
        
        for cache_key, elevations in self.temp_elevation_cache.items():
            elevations_str_keys = {str(k): v for k, v in elevations.items()}
            
            # Check if this tile exists in main cache
            if cache_key in self.elevation_cache['tiles']:
                old_count = len(self.elevation_cache['tiles'][cache_key].get('elevations', {}))
                overwritten_count += 1
                print(f"  Overwriting: {cache_key[:40]}... ({old_count} -> {len(elevations_str_keys)} points)")
            else:
                new_count += 1
                print(f"  Adding new: {cache_key[:40]}... ({len(elevations_str_keys)} points)")
            
            total_points_updated += len(elevations_str_keys)
            
            # Get tile info from temp cache or create minimal info
            tile_info = {
                'elevations': elevations_str_keys,
                'data_type': 'lidar',
                'timestamp': time.time(),
                'num_points': len(elevations_str_keys),
                'override_applied': True
            }
            
            # Preserve some fields from original if they exist
            if cache_key in self.elevation_cache['tiles']:
                original = self.elevation_cache['tiles'][cache_key]
                tile_info['tile_name'] = original.get('tile_name', 'Unknown')
                tile_info['tile_bbox'] = original.get('tile_bbox')
            
            self.elevation_cache['tiles'][cache_key] = tile_info
        
        print(f"\nMerge Summary:")
        print(f"  Tiles overwritten: {overwritten_count}")
        print(f"  New tiles added: {new_count}")
        print(f"  Total points updated: {total_points_updated}")
        
        # Save the merged cache
        print(f"  Saving merged cache...")
        self._save_elevation_cache()
        
        # Clear temporary cache
        self.temp_elevation_cache.clear()
        print(f"  ✓ Override cache merge complete")

# ==================== ELEVATION DATA DOWNLOAD ====================

class ElevationDataDownloader:
    """Download elevation data files"""
    
    def __init__(self, data_dir: Path = DATA_DIR):
        self.data_dir = data_dir
        self.session = requests.Session()
        self.session.headers.update({'User-Agent': 'Python Elevation Downloader 1.0'})
    
    def download_tile(self, tile: ElevationTile) -> bool:
        """Download a single tile if not already cached"""
        if not tile.url:
            return False
        
        filename = Path(tile.url).name
        if not filename:
            filename = f"{tile.data_type}_{tile.name.replace(' ', '_')[:50]}.tif"
        
        local_path = self.data_dir / tile.data_type / filename
        local_path.parent.mkdir(parents=True, exist_ok=True)
        
        if local_path.exists() and local_path.stat().st_size > 1000:
            tile.local_path = local_path
            return True
        
        try:
            print(f"    Checking {filename}...")
            response = self.session.head(tile.url, timeout=10, allow_redirects=True)
            
            if response.status_code != 200:
                print(f"    Error: HTTP {response.status_code} for {filename}")
                return False
                
            if 'content-length' in response.headers:
                size_mb = int(response.headers['content-length']) / (1024 * 1024)
                if size_mb > MAX_FILE_SIZE_MB:
                    print(f"    Skipping {filename}: {size_mb:.1f} MB exceeds limit")
                    return False
                print(f"    Downloading {filename} ({size_mb:.1f} MB)...")
            else:
                print(f"    Downloading {filename} (size unknown)...")
            
            response = self.session.get(tile.url, stream=True, timeout=30, allow_redirects=True)
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            downloaded = 0
            
            with open(local_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
            
            if local_path.stat().st_size < 1000:
                print(f"    Error: Downloaded file too small ({local_path.stat().st_size} bytes)")
                local_path.unlink()
                return False
            
            print(f"    Downloaded: {filename} ({local_path.stat().st_size / 1024 / 1024:.1f} MB)")
            tile.local_path = local_path
            return True
            
        except Exception as e:
            print(f"    Error downloading {filename}: {str(e)[:100]}")
            if local_path.exists():
                local_path.unlink()
            return False
    
    def download_tiles_parallel(self, tiles: List[ElevationTile]) -> List[ElevationTile]:
        """Download multiple tiles in parallel"""
        downloaded_tiles = []
        
        with ThreadPoolExecutor(max_workers=MAX_DOWNLOAD_WORKERS) as executor:
            future_to_tile = {executor.submit(self.download_tile, tile): tile for tile in tiles}
            
            for future in tqdm(as_completed(future_to_tile), total=len(tiles), desc="Downloading tiles"):
                tile = future_to_tile[future]
                try:
                    if future.result():
                        downloaded_tiles.append(tile)
                except Exception as e:
                    print(f"    Download failed for {tile.name}: {str(e)[:100]}")
        
        return downloaded_tiles

# ==================== ELEVATION DATA EXTRACTION ====================

class ElevationExtractor:
    """Extract elevation values with robust caching"""
    
    def __init__(self, cache_dir: Path = None):
        self.cache_dir = cache_dir or Path("Elevation_Cache")
        self.override_cache = OVERRIDE_CACHE  # Use global config
        self.temp_elevation_cache = {}  # Temporary cache for override mode
        self.cache_dir.mkdir(exist_ok=True)
        
        # Use pickle with compression for elevation cache
        self.elevation_cache_file = self.cache_dir / "elevation_cache.pkl.gz"
        self.cache_manager = RobustCache(self.elevation_cache_file, use_compression=True)
        
        # Load cache
        self.elevation_cache = self._load_elevation_cache()
        self.cache_hits = 0
        self.cache_misses = 0
    
    def _load_elevation_cache(self) -> Dict:
        """Load elevation cache with robust error recovery"""
        data = self.cache_manager._load_with_recovery("elevation cache")
        
        if data and 'tiles' in data:
            print(f"   Elevation cache contains {len(data['tiles'])} cached tiles")
            return data
        else:
            return {'tiles': {}, 'last_updated': time.time()}
    
    def _save_elevation_cache(self):
        """Save elevation cache with automatic backup"""
        try:
            self.elevation_cache['last_updated'] = time.time()
            success = self.cache_manager._save_with_backup(
                self.elevation_cache, 
                f"elevation cache ({self.cache_hits} hits, {self.cache_misses} misses)"
            )
            if not success:
                print("   Warning: Could not save elevation cache, continuing anyway")
        except Exception as e:
            print(f"   Warning: Could not save elevation cache: {e}")
    
    def _get_tile_cache_key(self, tile: ElevationTile) -> str:
        """Generate unique cache key for a tile's elevation data"""
        if tile.url:
            return f"{tile.data_type}_{hashlib.md5(tile.url.encode()).hexdigest()}"
        else:
            bbox_str = f"{tile.bbox[0]:.4f}_{tile.bbox[1]:.4f}_{tile.bbox[2]:.4f}_{tile.bbox[3]:.4f}"
            return f"{tile.data_type}_{bbox_str}"
    
    def _check_elevation_cache(self, tile: ElevationTile, points: List[RoutePoint]) -> Optional[Dict[int, float]]:
        """Check if elevation data for this tile is already cached"""
        cache_key = self._get_tile_cache_key(tile)

        # If override mode is active and this is a LiDAR tile, skip cache
        if self.override_cache and tile.data_type == 'lidar':
            # Check temporary cache first
            if cache_key in self.temp_elevation_cache:
                print(f"      Using temporary cache for {tile.name}")
                return self.temp_elevation_cache[cache_key]
            
            print(f"      Override mode: skipping main cache for LiDAR tile {tile.name}")
            self.cache_misses += 1
            return None
        
        if 'tiles' not in self.elevation_cache:
            self.elevation_cache['tiles'] = {}
        
        if cache_key in self.elevation_cache['tiles']:
            cached_data = self.elevation_cache['tiles'][cache_key]
            elevations = {}
            
            # For LiDAR tiles, we need to check all points that could potentially be in this tile
            # not just the ones in tile.covered_points (which might not be set correctly)
            points_to_check = []
            
            if tile.data_type == 'lidar':
                # For LiDAR, check all points within the tile's bounding box
                for point in points:
                    if (tile.bbox[0] <= point.lon <= tile.bbox[2] and 
                        tile.bbox[1] <= point.lat <= tile.bbox[3]):
                        points_to_check.append(point)
            else:
                # For other data types, use the original logic
                points_to_check = [p for p in points if p.index in tile.covered_points]
            
            # Check cache for each relevant point
            for point in points_to_check:
                point_key = str(point.index)
                if point_key in cached_data.get('elevations', {}):
                    elevations[point.index] = cached_data['elevations'][point_key]
            
            if elevations:
                self.cache_hits += 1
                # Log cache hit details for debugging
                if tile.data_type == 'lidar':
                    print(f"      Cache HIT for {tile.name}: {len(elevations)} cached elevations found")
                return elevations
        
        self.cache_misses += 1
        return None
    
    def _save_to_elevation_cache(self, tile: ElevationTile, elevations: Dict[int, float]):
        """Save extracted elevations to cache"""
        cache_key = self._get_tile_cache_key(tile)
        
        # In override mode for LiDAR, save to temporary cache
        if self.override_cache and tile.data_type == 'lidar':
            print(f"      Saving to temporary cache: {tile.name} ({len(elevations)} points)")
            self.temp_elevation_cache[cache_key] = elevations
            return  # Don't save to main cache yet
        
        if 'tiles' not in self.elevation_cache:
            self.elevation_cache['tiles'] = {}
        
        elevations_str_keys = {str(k): v for k, v in elevations.items()}
        
        # Check if we're updating an existing cache entry
        is_update = cache_key in self.elevation_cache['tiles']
        
        self.elevation_cache['tiles'][cache_key] = {
            'tile_name': tile.name,
            'tile_bbox': tile.bbox,
            'data_type': tile.data_type,
            'elevations': elevations_str_keys,
            'timestamp': time.time(),
            'num_points': len(elevations_str_keys)
        }
        
        # Log cache save for debugging
        if tile.data_type == 'lidar':
            action = "Updated" if is_update else "Saved"
            print(f"      {action} cache for {tile.name}: {len(elevations_str_keys)} elevations (key: {cache_key[:20]}...)")
        
        # Periodically save cache (every 10 new tiles)
        if len(self.elevation_cache['tiles']) % 10 == 0:
            self._save_elevation_cache()
    
    def extract_from_lidar(self, tile: ElevationTile, points: List[RoutePoint]) -> Dict[int, float]:
        """Extract elevations from LiDAR point cloud with proper coordinate transformation"""

        # === ADD THIS DEBUG SECTION ===
        # Debug: Show what we're looking for
        cache_key = self._get_tile_cache_key(tile)
        points_in_bbox = sum(1 for p in points if 
                            tile.bbox[0] <= p.lon <= tile.bbox[2] and 
                            tile.bbox[1] <= p.lat <= tile.bbox[3])
        print(f"    Checking cache for {tile.name} (key: {cache_key[:30]}...)")
        print(f"      - Points in bbox: {points_in_bbox}")
        print(f"      - Tile covered points: {len(tile.covered_points)}")
        # === END DEBUG SECTION ===
        
        cached_elevations = self._check_elevation_cache(tile, points)
        if cached_elevations is not None:
            print(f"    ✓ Using cached elevations for {tile.name}: {len(cached_elevations)} points")  # Enhanced message
            return cached_elevations
        
        print(f"    ✗ Cache miss for {tile.name}, extracting from file...")  # Enhanced message
        
        if not LASPY_AVAILABLE:
            print("    Warning: laspy not available, skipping LiDAR extraction")
            return {}
        
        if not PYPROJ_AVAILABLE:
            print("    ERROR: pyproj is required for coordinate transformation!")
            print("    Install with: pip install pyproj")
            return {}
            
        if not tile.local_path or not tile.local_path.exists():
            print(f"    Warning: File not found: {tile.local_path}")
            return {}
        
        try:
            # Try to read the LAS/LAZ file
            try:
                with laspy.open(tile.local_path) as f:
                    # Try reading in chunks if full read fails
                    try:
                        las = f.read()
                    except Exception as read_error:
                        # If reading all points fails, try reading the header and a subset
                        print(f"      Warning: Could not read all points from {tile.local_path.name}")
                        print(f"      Error: {str(read_error)[:100]}")
                        
                        # Try alternative reading approach - read header only first
                        try:
                            # Create a new file handle and read in smaller chunks
                            with laspy.open(tile.local_path) as f_retry:
                                header = f_retry.header
                                total_points = header.point_count
                                
                                # Try to read a subset of points
                                max_points_to_read = min(100000, total_points)  # Read up to 100k points
                                print(f"      Attempting to read first {max_points_to_read} points of {total_points} total")
                                
                                las = f_retry.read_points(max_points_to_read)
                                print(f"      Successfully read {len(las.x)} points")
                        except:
                            # If even subset reading fails, skip this file
                            print(f"      ERROR: File appears corrupted, skipping: {tile.local_path.name}")
                            
                            # Optionally delete corrupted file for re-download
                            if AUTO_DELETE_CORRUPTED and tile.local_path.exists():
                                try:
                                    tile.local_path.unlink()
                                    print(f"      Deleted corrupted file for re-download: {tile.local_path.name}")
                                except:
                                    print(f"      Could not delete corrupted file: {tile.local_path.name}")
                            
                            return {}
            except Exception as open_error:
                # Handle LAZ decompression errors
                if "LaszipError" in str(type(open_error).__name__):
                    print(f"      ERROR: LAZ decompression failed for {tile.local_path.name}")
                    print(f"      This file may be corrupted or use an unsupported compression")
                    
                    # Optionally delete corrupted file for re-download
                    if AUTO_DELETE_CORRUPTED and tile.local_path.exists():
                        try:
                            tile.local_path.unlink()
                            print(f"      Deleted corrupted file for re-download: {tile.local_path.name}")
                            print(f"      Re-run the script to download a fresh copy")
                        except:
                            print(f"      Could not delete corrupted file: {tile.local_path.name}")
                    else:
                        print(f"      Consider deleting and re-downloading: {tile.local_path}")
                else:
                    print(f"      ERROR: Could not open {tile.local_path.name}: {str(open_error)[:100]}")
                return {}
                
            lidar_x = las.x
            lidar_y = las.y
            lidar_z = las.z  # Keep in feet - no conversion
            
            if len(lidar_x) == 0:
                print(f"      Warning: No points in {tile.local_path.name}")
                return {}
            
            # Detect CRS and create transformer
            lidar_crs = detect_lidar_crs(lidar_x, lidar_y, las.header, filename=tile.local_path.name)
            
            if lidar_crs is None:
                print(f"      ERROR: Could not detect coordinate system for {tile.local_path.name}")
                print(f"      Defaulting to Ohio State Plane South (US Survey Feet) - EPSG:3735")
                lidar_crs = CRS.from_epsg(3735)
            
            # Create transformer from WGS84 to LiDAR CRS
            from pyproj import Transformer
            wgs84_crs = CRS.from_epsg(4326)
            
            # Check if the CRS has a valid EPSG code
            epsg_code = lidar_crs.to_epsg()
            if epsg_code is None:
                print(f"      Warning: CRS has no EPSG code, assuming Ohio State Plane South")
                lidar_crs = CRS.from_epsg(3735)
                epsg_code = 3735
            
            if epsg_code == 4326:
                # No transformation needed
                transformer = None
                print("      No coordinate transformation needed (both WGS84)")
            else:
                try:
                    transformer = Transformer.from_crs(
                        wgs84_crs,
                        lidar_crs,
                        always_xy=True  # Use (lon, lat) order
                    )
                    print(f"      Created transformer: WGS84 → EPSG:{epsg_code}")
                except Exception as e:
                    print(f"      Error creating transformer: {e}")
                    print(f"      Defaulting to Ohio State Plane South")
                    lidar_crs = CRS.from_epsg(3735)
                    transformer = Transformer.from_crs(
                        wgs84_crs,
                        lidar_crs,
                        always_xy=True
                    )
            
            # Build KD-tree
            lidar_coords = np.column_stack([lidar_x, lidar_y])
            tree = cKDTree(lidar_coords)
            
            elevations = {}
            points_processed = 0
            transform_errors = 0
            
            for point in points:
                if point.index not in tile.covered_points:
                    continue
                
                try:
                    if transformer is not None:
                        # Transform point from WGS84 to LiDAR CRS
                        point_x, point_y = transformer.transform(point.lon, point.lat)
                    else:
                        # Both in WGS84, use as-is
                        point_x = point.lon
                        point_y = point.lat
                    
                    # Find nearest LiDAR points
                    distances, indices = tree.query(
                        [point_x, point_y],
                        k=min(LIDAR_MAX_POINTS, len(lidar_x))
                    )
                    
                    # Handle single vs multiple results
                    if not isinstance(distances, np.ndarray):
                        distances = np.array([distances])
                        indices = np.array([indices])
                    
                    # Filter by search radius (converting feet to match coordinate system if needed)
                    # Since we're in State Plane (feet), use feet directly
                    search_radius_feet = LIDAR_SEARCH_RADIUS_M * 3.28084  # Convert meters to feet
                    mask = distances < search_radius_feet
                    if mask.any():
                        valid_distances = distances[mask]
                        valid_indices = indices[mask]
                        
                        if len(valid_distances) == 1:
                            elevations[point.index] = float(lidar_z[valid_indices[0]])  # Already in feet
                        else:
                            # Inverse distance weighting
                            weights = 1.0 / (valid_distances ** LIDAR_IDW_POWER)
                            weights /= weights.sum()
                            elevation = np.sum(lidar_z[valid_indices] * weights)
                            elevations[point.index] = float(elevation)  # Already in feet
                        points_processed += 1
                
                except Exception as e:
                    transform_errors += 1
                    continue
            
            if points_processed > 0:
                print(f"      Extracted {points_processed} elevations from {tile.local_path.name}")
            if transform_errors > 0:
                print(f"      Warning: {transform_errors} coordinate transformation errors")
            
            if elevations:
                self._save_to_elevation_cache(tile, elevations)
            
            return elevations
            
        except Exception as e:
            print(f"    Error processing LiDAR file {tile.local_path.name}: {str(e)[:200]}")
            if "memory" in str(e).lower():
                print(f"      Possible memory issue - file may be too large")
            return {}
    
    def extract_from_dem(self, tile: ElevationTile, points: List[RoutePoint]) -> Dict[int, float]:
        """Extract elevations from DEM raster"""
        cached_elevations = self._check_elevation_cache(tile, points)
        if cached_elevations is not None:
            return cached_elevations
        
        if not RASTERIO_AVAILABLE:
            print("    Warning: rasterio not available, skipping DEM extraction")
            return {}
            
        if not tile.local_path or not tile.local_path.exists():
            return {}
        
        try:
            with rasterio.open(tile.local_path) as src:
                elevations = {}
                
                for point in points:
                    if point.index in tile.covered_points:
                        row, col = src.index(point.lon, point.lat)
                        
                        if 0 <= row < src.height and 0 <= col < src.width:
                            elevation = src.read(1)[row, col]
                            
                            if elevation != src.nodata and not np.isnan(elevation):
                                # Check if elevation is in meters and convert to feet
                                # Most DEMs are in meters, but check the CRS
                                if hasattr(src, 'crs') and src.crs:
                                    crs_str = str(src.crs).lower()
                                    if 'meter' in crs_str or 'metre' in crs_str:
                                        elevation = elevation * 3.28084  # Convert meters to feet
                                elevations[point.index] = float(elevation)
                
                if elevations:
                    self._save_to_elevation_cache(tile, elevations)
                
                return elevations
                
        except Exception as e:
            print(f"    Error processing DEM file {tile.local_path.name}: {str(e)[:100]}")
            return {}
    
    def extract_from_opentopo(self, points: List[RoutePoint]) -> Dict[int, float]:
        """Extract elevations from OpenTopography API"""
        elevations = {}
        session = requests.Session()
        
        for point in points:
            if point.data_source == 'opentopo':
                try:
                    params = {
                        'x': point.lon,
                        'y': point.lat,
                        'units': 'Feet',  # Request feet directly
                        'output': 'json'
                    }
                    
                    response = session.get(USGS_ELEVATION_API, params=params, timeout=10)
                    response.raise_for_status()
                    data = response.json()
                    
                    if 'value' in data:
                        elevation = float(data['value'])
                        if elevation > -9999:
                            elevations[point.index] = elevation  # Already in feet
                    
                    time.sleep(0.1)
                    
                except Exception as e:
                    print(f"    Error fetching elevation for point {point.index}: {str(e)[:50]}")
        
        return elevations
    
    def extract_elevations(self, tiles_by_type: Dict[str, List[ElevationTile]], 
                          points: List[RoutePoint]) -> List[RoutePoint]:
        """Extract elevations from all data sources"""
        print("\n" + "="*60)
        print("EXTRACTING ELEVATION DATA")
        print("="*60)
        
        for data_type in ['lidar', 'dem_sub1m', 'dem_1m', 'dem_10m']:
            if data_type not in tiles_by_type:
                continue
            
            tiles = tiles_by_type[data_type]
            print(f"\nProcessing {data_type} tiles...")
            
            for tile in tqdm(tiles, desc=f"  Extracting from {data_type}"):
                if data_type == 'lidar':
                    elevations = self.extract_from_lidar(tile, points)
                else:
                    elevations = self.extract_from_dem(tile, points)
                
                for idx, elevation in elevations.items():
                    if points[idx].elevation is None:
                        points[idx].elevation = elevation
        
        if 'opentopo' in tiles_by_type:
            print("\nFetching OpenTopography elevations...")
            opentopo_points = [p for p in points if p.data_source == 'opentopo']
            if opentopo_points:
                elevations = self.extract_from_opentopo(opentopo_points)
                for idx, elevation in elevations.items():
                    points[idx].elevation = elevation
        
        self._interpolate_gaps(points)
        self._save_elevation_cache()
        
        covered = sum(1 for p in points if p.elevation is not None)
        print(f"\nElevation extraction complete:")
        print(f"  {covered}/{len(points)} points have elevation data ({covered/len(points)*100:.1f}%)")
        print(f"  Cache performance: {self.cache_hits} hits, {self.cache_misses} misses")
        
        return points

    def merge_override_cache(self):
        """Merge temporary cache into main cache, overwriting conflicts"""
        if not self.override_cache or not self.temp_elevation_cache:
            return
        
        print("\n" + "="*60)
        print("MERGING OVERRIDE CACHE")
        print("="*60)
        
        if 'tiles' not in self.elevation_cache:
            self.elevation_cache['tiles'] = {}
        
        overwritten_count = 0
        new_count = 0
        total_points_updated = 0
        
        for cache_key, elevations in self.temp_elevation_cache.items():
            elevations_str_keys = {str(k): v for k, v in elevations.items()}
            
            # Check if this tile exists in main cache
            if cache_key in self.elevation_cache['tiles']:
                old_count = len(self.elevation_cache['tiles'][cache_key].get('elevations', {}))
                overwritten_count += 1
                print(f"  Overwriting: {cache_key[:40]}... ({old_count} -> {len(elevations_str_keys)} points)")
            else:
                new_count += 1
                print(f"  Adding new: {cache_key[:40]}... ({len(elevations_str_keys)} points)")
            
            total_points_updated += len(elevations_str_keys)
            
            # Get tile info from temp cache or create minimal info
            tile_info = {
                'elevations': elevations_str_keys,
                'data_type': 'lidar',
                'timestamp': time.time(),
                'num_points': len(elevations_str_keys),
                'override_applied': True
            }
            
            # Preserve some fields from original if they exist
            if cache_key in self.elevation_cache['tiles']:
                original = self.elevation_cache['tiles'][cache_key]
                tile_info['tile_name'] = original.get('tile_name', 'Unknown')
                tile_info['tile_bbox'] = original.get('tile_bbox')
            
            self.elevation_cache['tiles'][cache_key] = tile_info
        
        print(f"\nMerge Summary:")
        print(f"  Tiles overwritten: {overwritten_count}")
        print(f"  New tiles added: {new_count}")
        print(f"  Total points updated: {total_points_updated}")
        
        # Save the merged cache
        print(f"  Saving merged cache...")
        self._save_elevation_cache()
        
        # Clear temporary cache
        self.temp_elevation_cache.clear()
        print(f"  ✓ Override cache merge complete")
    
    def _interpolate_gaps(self, points: List[RoutePoint]):
        """Interpolate elevation for points with missing data"""
        for i, point in enumerate(points):
            if point.elevation is None:
                before_idx = None
                after_idx = None
                
                for j in range(i - 1, -1, -1):
                    if points[j].elevation is not None:
                        before_idx = j
                        break
                
                for j in range(i + 1, len(points)):
                    if points[j].elevation is not None:
                        after_idx = j
                        break
                
                if before_idx is not None and after_idx is not None:
                    dist_before = point.distance_m - points[before_idx].distance_m
                    dist_after = points[after_idx].distance_m - point.distance_m
                    total_dist = dist_before + dist_after
                    
                    if total_dist > 0:
                        weight_before = dist_after / total_dist
                        weight_after = dist_before / total_dist
                        point.elevation = (
                            points[before_idx].elevation * weight_before +
                            points[after_idx].elevation * weight_after
                        )
                        point.is_interpolated = True
                elif before_idx is not None:
                    point.elevation = points[before_idx].elevation
                    point.is_interpolated = True
                elif after_idx is not None:
                    point.elevation = points[after_idx].elevation
                    point.is_interpolated = True

# ==================== ELEVATION FILTERING AND SMOOTHING ====================

class ElevationProcessor:
    """Process elevation data to remove noise and handle bridges/tunnels"""
    
    def __init__(self):
        self.bridge_tunnel_segments = []
        self.outlier_points = []
        self.manual_bridge_segments = []  # Store manual bridge/tunnel segments
        
    def apply_bridge_tunnel_leveling(self, points: List[RoutePoint], 
                                bridge_segments: List[BridgeTunnelSegment]) -> List[RoutePoint]:
        """Apply linear interpolation for ALL points between bridge/tunnel endpoints"""
        if not bridge_segments:
            return points
        
        print(f"\n  Applying bridge/tunnel leveling from {len(bridge_segments)} segments...")
        
        total_leveled = 0
        segments_skipped = 0
        
        for segment in bridge_segments:
            # Find the route points closest to segment start and end
            start_idx = None
            end_idx = None
            min_start_dist = float('inf')
            min_end_dist = float('inf')
            
            # Find closest points to the bridge/tunnel endpoints
            for i, point in enumerate(points):
                start_dist = haversine_distance(point.lat, point.lon, 
                                              segment.start_lat, segment.start_lon)
                end_dist = haversine_distance(point.lat, point.lon, 
                                            segment.end_lat, segment.end_lon)
                
                if start_dist < min_start_dist:
                    min_start_dist = start_dist
                    start_idx = i
                
                if end_dist < min_end_dist:
                    min_end_dist = end_dist
                    end_idx = i
            
            # Check if we found valid endpoints
            if start_idx is None or end_idx is None:
                print(f"    Warning: Could not find endpoints for segment '{segment.name}'")
                segments_skipped += 1
                continue
            
            # CHECK DISTANCE THRESHOLD - Skip if either endpoint is > 10m away
            DISTANCE_THRESHOLD = 10.0  # meters
            if min_start_dist > DISTANCE_THRESHOLD or min_end_dist > DISTANCE_THRESHOLD:
                print(f"    Skipping segment '{segment.name}':")
                print(f"      Start point distance: {min_start_dist:.1f}m")
                print(f"      End point distance: {min_end_dist:.1f}m")
                print(f"      Exceeds {DISTANCE_THRESHOLD}m threshold - likely misalignment")
                segments_skipped += 1
                continue
            
            # Ensure start comes before end
            if start_idx > end_idx:
                start_idx, end_idx = end_idx, start_idx
            
            if start_idx == end_idx:
                print(f"    Warning: Start and end are the same point for segment '{segment.name}'")
                segments_skipped += 1
                continue
            
            # Get elevations at endpoints
            start_elevation = points[start_idx].elevation
            end_elevation = points[end_idx].elevation
            
            if start_elevation is None or end_elevation is None:
                print(f"    Warning: Missing elevation data for segment '{segment.name}'")
                segments_skipped += 1
                continue
            
            # Get distances for interpolation
            start_distance = points[start_idx].distance_m
            end_distance = points[end_idx].distance_m
            total_distance = end_distance - start_distance
            
            if total_distance <= 0:
                print(f"    Warning: Invalid distance for segment '{segment.name}'")
                segments_skipped += 1
                continue
            
            # Store original elevations for reporting
            original_elevations = []
            
            # IMPORTANT: Level ALL points between start and end indices
            points_to_level = 0
            for idx in range(start_idx, end_idx + 1):
                if points[idx].elevation is not None:
                    original_elevations.append(points[idx].elevation)
                
                # Don't modify the endpoints themselves (optional - remove if you want endpoints leveled too)
                if idx == start_idx or idx == end_idx:
                    continue
                
                # Calculate interpolated elevation based on distance along route
                point = points[idx]
                distance_along = point.distance_m - start_distance
                fraction = distance_along / total_distance
                
                # Linear interpolation
                new_elevation = start_elevation + fraction * (end_elevation - start_elevation)
                
                # Apply the leveling
                point.elevation = new_elevation
                point.is_interpolated = True
                points_to_level += 1
                total_leveled += 1
            
            # Report results for successfully leveled segments
            if original_elevations:
                original_min = min(original_elevations)
                original_max = max(original_elevations)
                original_range = original_max - original_min
            else:
                original_range = 0
            
            print(f"    ✓ Leveled segment '{segment.name}':")
            print(f"      Points leveled: {points_to_level} (indices {start_idx} to {end_idx})")
            print(f"      Match distances: {min_start_dist:.1f}m (start), {min_end_dist:.1f}m (end)")
            print(f"      Distance: {total_distance:.1f} m")
            print(f"      Elevation at start: {start_elevation:.1f} ft")
            print(f"      Elevation at end: {end_elevation:.1f} ft")
            print(f"      Gradient: {(end_elevation - start_elevation) / (total_distance * 3.28084) * 100:.2f}%")
            if original_range > 0:
                print(f"      Original elevation range: {original_range:.1f} ft → flattened")
            
            # Validation check
            if points_to_level == 0:
                print(f"      WARNING: No points were actually leveled!")
            elif points_to_level < (end_idx - start_idx - 1):
                print(f"      WARNING: Expected to level {end_idx - start_idx - 1} points but only leveled {points_to_level}")
        
        # Final summary
        print(f"\n  Bridge/Tunnel Leveling Summary:")
        print(f"    Total segments provided: {len(bridge_segments)}")
        print(f"    Segments successfully leveled: {len(bridge_segments) - segments_skipped}")
        print(f"    Segments skipped (>10m away): {segments_skipped}")
        if total_leveled > 0:
            print(f"    Total points leveled: {total_leveled}")
        else:
            print(f"    WARNING: No points were leveled! Check alignment between route and bridge/tunnel segments.")
        
        # Store for reporting
        self.manual_bridge_segments = bridge_segments
        
        return points
    
    def filter_bridges_and_tunnels(self, points: List[RoutePoint]) -> List[RoutePoint]:
        """Detect and filter elevation spikes from bridges, tunnels, and vegetation"""
        if not BRIDGE_TUNNEL_FILTER_ENABLED:
            return points
        
        print("\n  Filtering bridges, tunnels, and outliers...")
        
        elevations = np.array([p.elevation if p.elevation is not None else np.nan 
                               for p in points])
        
        # Skip if too few valid elevations
        valid_mask = ~np.isnan(elevations)
        if np.sum(valid_mask) < 10:
            return points
        
        # Step 1: Detect sudden elevation changes (potential bridges/overpasses)
        outliers = self._detect_elevation_outliers(points, elevations)
        
        # Step 2: Detect unrealistic gradients
        gradient_issues = self._detect_gradient_issues(points, elevations)
        
        # Step 3: Detect bridge/tunnel patterns
        structures = self._detect_structures(points, elevations)
        
        # Combine all detected issues
        all_issues = set(outliers) | set(gradient_issues) | set(structures)
        
        if all_issues:
            print(f"    Found {len(all_issues)} problematic points:")
            print(f"      - Elevation outliers: {len(outliers)}")
            print(f"      - Gradient issues: {len(gradient_issues)}")
            print(f"      - Bridge/tunnel patterns: {len(structures)}")
            
            # Interpolate through problematic segments
            self._fix_problematic_segments(points, all_issues)
            
            # Store for reporting
            self.outlier_points = list(all_issues)
        
        return points
    
    def _detect_elevation_outliers(self, points: List[RoutePoint], elevations: np.ndarray) -> List[int]:
        """Detect sudden elevation jumps that might be bridges or data errors"""
        outliers = []
        
        # Calculate local elevation changes
        for i in range(1, len(elevations) - 1):
            if np.isnan(elevations[i]):
                continue
                
            # Look at nearby points
            window_indices = []
            for j in range(max(0, i - 5), min(len(elevations), i + 6)):
                if j != i and not np.isnan(elevations[j]):
                    window_indices.append(j)
            
            if len(window_indices) < 3:
                continue
            
            local_elevations = elevations[window_indices]
            local_median = np.median(local_elevations)
            
            # Check if current point is an outlier
            elevation_diff = abs(elevations[i] - local_median)
            if elevation_diff > OUTLIER_THRESHOLD_FT:
                outliers.append(i)
                
                # Also check if this might be a road overpass/underpass pattern
                # (elevation spike up then immediately down, or vice versa)
                if i > 0 and i < len(elevations) - 1:
                    if not np.isnan(elevations[i-1]) and not np.isnan(elevations[i+1]):
                        before_diff = elevations[i] - elevations[i-1]
                        after_diff = elevations[i+1] - elevations[i]
                        
                        # Classic overpass pattern: sudden rise and fall
                        if (abs(before_diff) > 20 and abs(after_diff) > 20 and 
                            before_diff * after_diff < 0):  # Opposite signs
                            if i-1 not in outliers:
                                outliers.append(i-1)
                            if i+1 not in outliers:
                                outliers.append(i+1)
        
        return outliers
    
    def _detect_gradient_issues(self, points: List[RoutePoint], elevations: np.ndarray) -> List[int]:
        """Detect unrealistic gradients for rail"""
        issues = []
        
        for i in range(1, len(elevations)):
            if np.isnan(elevations[i]) or np.isnan(elevations[i-1]):
                continue
            
            # Calculate gradient
            elevation_change = elevations[i] - elevations[i-1]
            distance = points[i].distance_m - points[i-1].distance_m
            
            if distance > 0:
                gradient_percent = abs(elevation_change / (distance * 3.28084)) * 100  # Convert m to ft
                
                if gradient_percent > MAX_GRADIENT_PERCENT:
                    issues.append(i)
                    if i-1 not in issues:
                        issues.append(i-1)
        
        return issues
    
    def _detect_structures(self, points: List[RoutePoint], elevations: np.ndarray) -> List[int]:
        """Detect bridge and tunnel patterns in elevation data"""
        structures = []
        
        # Look for characteristic patterns
        window_points = int(BRIDGE_TUNNEL_WINDOW_M / INTERPOLATION_SPACING_M)
        
        for i in range(window_points, len(elevations) - window_points):
            if np.isnan(elevations[i]):
                continue
            
            # Get window around point
            window_start = i - window_points
            window_end = i + window_points + 1
            window_elevations = elevations[window_start:window_end]
            
            # Skip if too many missing values
            valid_mask = ~np.isnan(window_elevations)
            if np.sum(valid_mask) < len(window_elevations) * 0.5:
                continue
            
            # Check for bridge pattern: elevated section with approaches
            center_elev = elevations[i]
            edge_elevations = np.concatenate([
                window_elevations[:window_points//2],
                window_elevations[-window_points//2:]
            ])
            edge_elevations = edge_elevations[~np.isnan(edge_elevations)]
            
            if len(edge_elevations) > 0:
                edge_median = np.median(edge_elevations)
                
                # Bridge: center significantly higher than edges
                if center_elev - edge_median > 30:  # 30 feet higher suggests a bridge
                    for j in range(max(0, i-2), min(len(elevations), i+3)):
                        structures.append(j)
                
                # Tunnel: center significantly lower than edges (less common in LiDAR)
                elif edge_median - center_elev > 30:
                    for j in range(max(0, i-2), min(len(elevations), i+3)):
                        structures.append(j)
        
        return list(set(structures))
    
    def _fix_problematic_segments(self, points: List[RoutePoint], problematic_indices: Set[int]):
        """Interpolate through problematic elevation segments"""
        # Sort indices
        problem_indices = sorted(problematic_indices)
        
        # Group consecutive problematic points into segments
        segments = []
        current_segment = [problem_indices[0]]
        
        for i in range(1, len(problem_indices)):
            if problem_indices[i] == problem_indices[i-1] + 1:
                current_segment.append(problem_indices[i])
            else:
                segments.append(current_segment)
                current_segment = [problem_indices[i]]
        segments.append(current_segment)
        
        # Fix each segment
        for segment in segments:
            start_idx = segment[0]
            end_idx = segment[-1]
            
            # Find good points before and after segment
            before_idx = None
            for i in range(start_idx - 1, -1, -1):
                if i not in problematic_indices and points[i].elevation is not None:
                    before_idx = i
                    break
            
            after_idx = None
            for i in range(end_idx + 1, len(points)):
                if i not in problematic_indices and points[i].elevation is not None:
                    after_idx = i
                    break
            
            # Interpolate through the segment
            if before_idx is not None and after_idx is not None:
                # Linear interpolation based on distance
                for idx in segment:
                    dist_from_before = points[idx].distance_m - points[before_idx].distance_m
                    total_dist = points[after_idx].distance_m - points[before_idx].distance_m
                    
                    if total_dist > 0:
                        fraction = dist_from_before / total_dist
                        points[idx].elevation = (
                            points[before_idx].elevation * (1 - fraction) +
                            points[after_idx].elevation * fraction
                        )
                        # Mark as interpolated due to filtering
                        points[idx].is_interpolated = True
        
        # Store segment info
        self.bridge_tunnel_segments = segments
    
    def smooth_elevations(self, points: List[RoutePoint]) -> List[RoutePoint]:
        """Apply smoothing with cosine-based transitions at manual segment boundaries"""
        if not SMOOTHING_ENABLED:
            return points
        
        print(f"\n  Applying {SMOOTHING_METHOD} smoothing with cosine transitions...")
        
        # Extract elevations
        elevations = np.array([p.elevation if p.elevation is not None else np.nan 
                              for p in points])
        
        # First apply general smoothing to the entire profile
        smoothed_elevations = elevations.copy()
        valid_mask = ~np.isnan(elevations)
        
        if np.sum(valid_mask) > SMOOTHING_WINDOW * 2:
            # Create a fully smoothed version
            x = np.arange(len(elevations))
            valid_x = x[valid_mask]
            valid_y = elevations[valid_mask]
            
            if len(valid_x) > 1:
                # Interpolate NaN values
                interp_elevations = np.interp(x, valid_x, valid_y)
                
                # Apply smoothing to create baseline smooth profile
                if SMOOTHING_METHOD == 'gaussian':
                    sigma = SMOOTHING_WINDOW / 3.0
                    fully_smoothed = gaussian_filter1d(interp_elevations, sigma=sigma, mode='nearest')
                elif SMOOTHING_METHOD == 'median':
                    from scipy.signal import medfilt
                    window_size = SMOOTHING_WINDOW * 2 + 1
                    fully_smoothed = medfilt(interp_elevations, kernel_size=min(window_size, len(interp_elevations)))
                else:  # mean
                    from scipy.ndimage import uniform_filter1d
                    fully_smoothed = uniform_filter1d(interp_elevations, size=SMOOTHING_WINDOW * 2 + 1, mode='nearest')
                
                # Start with the fully smoothed version
                smoothed_elevations = fully_smoothed.copy()
        
        # Now handle manual segments with cosine transitions
        if hasattr(self, 'manual_bridge_segments') and self.manual_bridge_segments:
            # Extended transition zone (in points) - make it longer for smoother blend
            TRANSITION_LENGTH = 50  # About 500m at 10m spacing
            
            # Process each manual segment
            for segment in self.manual_bridge_segments:
                # Find the segment boundaries
                start_idx = None
                end_idx = None
                min_start_dist = float('inf')
                min_end_dist = float('inf')
                
                for i, point in enumerate(points):
                    start_dist = haversine_distance(point.lat, point.lon, 
                                                  segment.start_lat, segment.start_lon)
                    end_dist = haversine_distance(point.lat, point.lon, 
                                                segment.end_lat, segment.end_lon)
                    
                    if start_dist < min_start_dist:
                        min_start_dist = start_dist
                        start_idx = i
                    
                    if end_dist < min_end_dist:
                        min_end_dist = end_dist
                        end_idx = i
                
                if start_idx is not None and end_idx is not None:
                    if start_idx > end_idx:
                        start_idx, end_idx = end_idx, start_idx
                    
                    # Apply cosine transition at the start of the segment
                    start_transition_begin = max(0, start_idx - TRANSITION_LENGTH)
                    start_transition_end = min(len(elevations) - 1, start_idx + 5)  # Small overlap into segment
                    
                    if start_transition_begin < start_transition_end:
                        # Get the elevation targets
                        smooth_start_elev = smoothed_elevations[start_transition_begin] if start_transition_begin >= 0 else smoothed_elevations[0]
                        manual_start_elev = elevations[start_transition_end]
                        
                        # Apply cosine interpolation for start transition
                        for i in range(start_transition_begin, start_transition_end + 1):
                            # Calculate position in transition (0 to 1)
                            t = (i - start_transition_begin) / (start_transition_end - start_transition_begin)
                            
                            # Cosine interpolation (smooth S-curve)
                            cos_weight = (1 - np.cos(t * np.pi)) / 2
                            
                            # Blend between smooth terrain and manual elevation
                            smoothed_elevations[i] = (1 - cos_weight) * smooth_start_elev + cos_weight * manual_start_elev
                    
                    # Keep the core segment mostly as-is (light smoothing only)
                    core_start = start_transition_end + 1
                    core_end = max(start_idx, end_idx - 5)  # Leave room for end transition
                    
                    if core_start < core_end:
                        # Apply very light smoothing to core to remove any noise
                        core_elevations = elevations[core_start:core_end + 1]
                        if len(core_elevations) > 3:
                            # Very gentle gaussian smooth
                            core_smoothed = gaussian_filter1d(core_elevations, sigma=1.0, mode='nearest')
                            # Strong preference for original
                            smoothed_elevations[core_start:core_end + 1] = (
                                0.9 * elevations[core_start:core_end + 1] + 
                                0.1 * core_smoothed
                            )
                        else:
                            smoothed_elevations[core_start:core_end + 1] = elevations[core_start:core_end + 1]
                    
                    # Apply cosine transition at the end of the segment
                    end_transition_begin = max(start_idx, end_idx - 5)  # Small overlap from segment
                    end_transition_end = min(len(elevations) - 1, end_idx + TRANSITION_LENGTH)
                    
                    if end_transition_begin < end_transition_end:
                        # Get the elevation targets
                        manual_end_elev = elevations[end_transition_begin]
                        smooth_end_elev = fully_smoothed[end_transition_end] if end_transition_end < len(fully_smoothed) else fully_smoothed[-1]
                        
                        # Apply cosine interpolation for end transition
                        for i in range(end_transition_begin, end_transition_end + 1):
                            # Calculate position in transition (0 to 1)
                            t = (i - end_transition_begin) / (end_transition_end - end_transition_begin)
                            
                            # Cosine interpolation (smooth S-curve)
                            cos_weight = (1 - np.cos(t * np.pi)) / 2
                            
                            # Blend from manual elevation back to smooth terrain
                            smoothed_elevations[i] = (1 - cos_weight) * manual_end_elev + cos_weight * smooth_end_elev
                    
                    print(f"    Applied cosine transition for segment '{segment.name}':")
                    print(f"      Start transition: points {start_transition_begin} to {start_transition_end}")
                    print(f"      End transition: points {end_transition_begin} to {end_transition_end}")
        
        # Additional pass: Apply a secondary smoothing to the transitions themselves
        # This helps blend any remaining discontinuities
        if hasattr(self, 'manual_bridge_segments') and self.manual_bridge_segments:
            transition_zones = []
            for segment in self.manual_bridge_segments:
                # Re-find segments (this is redundant but keeps code cleaner)
                start_idx = None
                end_idx = None
                min_start_dist = float('inf')
                min_end_dist = float('inf')
                
                for i, point in enumerate(points):
                    start_dist = haversine_distance(point.lat, point.lon, 
                                                  segment.start_lat, segment.start_lon)
                    end_dist = haversine_distance(point.lat, point.lon, 
                                                segment.end_lat, segment.end_lon)
                    
                    if start_dist < min_start_dist:
                        min_start_dist = start_dist
                        start_idx = i
                    
                    if end_dist < min_end_dist:
                        min_end_dist = end_dist
                        end_idx = i
                
                if start_idx is not None and end_idx is not None:
                    if start_idx > end_idx:
                        start_idx, end_idx = end_idx, start_idx
                    
                    # Mark broader transition zones for final smoothing
                    transition_zones.append((max(0, start_idx - 60), min(len(elevations), start_idx + 10)))
                    transition_zones.append((max(0, end_idx - 10), min(len(elevations), end_idx + 60)))
            
            # Apply localized smoothing to transition zones only
            for zone_start, zone_end in transition_zones:
                if zone_end - zone_start > 5:
                    zone_elevations = smoothed_elevations[zone_start:zone_end]
                    # Gentle smooth to remove any remaining kinks
                    zone_smoothed = gaussian_filter1d(zone_elevations, sigma=3.0, mode='nearest')
                    # Blend with heavier weight on the already processed elevations
                    smoothed_elevations[zone_start:zone_end] = (
                        0.7 * smoothed_elevations[zone_start:zone_end] + 
                        0.3 * zone_smoothed
                    )
        
        # Apply smoothed elevations back to points
        changes = 0
        max_change = 0
        for i, point in enumerate(points):
            if not np.isnan(smoothed_elevations[i]) and point.elevation != smoothed_elevations[i]:
                old_val = point.elevation
                point.elevation = smoothed_elevations[i]
                change = abs(old_val - point.elevation)
                if change > 0.1:  # Only count significant changes
                    changes += 1
                    max_change = max(max_change, change)
        
        print(f"    Smoothing complete:")
        print(f"      Modified {changes} elevation values")
        print(f"      Maximum elevation change: {max_change:.1f} ft")
        print(f"      Transition method: Cosine interpolation (S-curve)")
        
        return points
    
    def _median_smooth(self, elevations: np.ndarray, window: int) -> np.ndarray:
        """Apply median smoothing"""
        smoothed = elevations.copy()
        
        for i in range(len(elevations)):
            if np.isnan(elevations[i]):
                continue
            
            # Get window indices
            start = max(0, i - window)
            end = min(len(elevations), i + window + 1)
            
            # Get valid values in window
            window_values = elevations[start:end]
            valid_values = window_values[~np.isnan(window_values)]
            
            if len(valid_values) > 0:
                smoothed[i] = np.median(valid_values)
        
        return smoothed
    
    def _gaussian_smooth(self, elevations: np.ndarray, window: int) -> np.ndarray:
        """Apply Gaussian smoothing"""
        # Fill NaN values with interpolation for smoothing
        valid_mask = ~np.isnan(elevations)
        if np.sum(valid_mask) < 2:
            return elevations
        
        # Interpolate NaN values
        x = np.arange(len(elevations))
        filled_elevations = np.interp(x, x[valid_mask], elevations[valid_mask])
        
        # Apply Gaussian filter
        sigma = window / 2.0  # Standard deviation
        smoothed = gaussian_filter1d(filled_elevations, sigma=sigma, mode='nearest')
        
        # Restore NaN values
        smoothed[~valid_mask] = np.nan
        
        return smoothed
    
    def _mean_smooth(self, elevations: np.ndarray, window: int) -> np.ndarray:
        """Apply moving average smoothing"""
        smoothed = elevations.copy()
        
        for i in range(len(elevations)):
            if np.isnan(elevations[i]):
                continue
            
            # Get window indices
            start = max(0, i - window)
            end = min(len(elevations), i + window + 1)
            
            # Get valid values in window
            window_values = elevations[start:end]
            valid_values = window_values[~np.isnan(window_values)]
            
            if len(valid_values) > 0:
                smoothed[i] = np.mean(valid_values)
        
        return smoothed

def clear_all_caches(cache_dir: Path):
    """Clear all cache files (use with caution)"""
    print("\nClearing all cache files...")
    
    cache_patterns = [
        "*.pkl", "*.pkl.gz", "*.json", "*.tmp",
        "global_chunk_*.pkl.gz"
    ]
    
    for pattern in cache_patterns:
        for cache_file in cache_dir.glob(pattern):
            try:
                cache_file.unlink()
                print(f"  Removed: {cache_file.name}")
            except Exception as e:
                print(f"  Error removing {cache_file.name}: {e}")
    
    print("Cache clearing complete")


def get_cache_statistics(cache_dir: Path):
    """Get statistics about current cache files"""
    print("\nCache Statistics:")
    print("="*60)
    
    total_size = 0
    cache_files = []
    
    for cache_file in cache_dir.glob("*.pkl*"):
        if cache_file.is_file():
            size_mb = cache_file.stat().st_size / (1024 * 1024)
            cache_files.append((cache_file.name, size_mb))
            total_size += size_mb
    
    # Sort by size
    cache_files.sort(key=lambda x: x[1], reverse=True)
    
    for name, size in cache_files:
        print(f"  {name}: {size:.2f} MB")
    
    print(f"\nTotal cache size: {total_size:.2f} MB")
    
    # Check for backups
    backup_dir = cache_dir / "cache_backups"
    if backup_dir.exists():
        backup_files = list(backup_dir.glob("*"))
        if backup_files:
            backup_size = sum(f.stat().st_size for f in backup_files) / (1024 * 1024)
            print(f"Backup files: {len(backup_files)} files, {backup_size:.2f} MB")

# ==================== MAIN PIPELINE ====================

def main():
    """Main pipeline for complete elevation data processing"""
    print("="*70)
    print("COMPLETE ELEVATION DATA PIPELINE WITH COORDINATE TRANSFORMATION")
    print("="*70)
    print(f"Input: {os.path.basename(INPUT_KMZ)}")
    print(f"Bridge/Tunnel KMZ: {os.path.basename(BRIDGE_TUNNEL_KMZ) if BRIDGE_TUNNEL_KMZ else 'None'}")
    print(f"Interpolation: {INTERPOLATION_SPACING_M}m")
    print(f"LiDAR interpolation: {LIDAR_INTERPOLATION_ENABLED}")
    if LIDAR_INTERPOLATION_ENABLED:
        print(f"  - Point spacing: {LIDAR_POINT_SPACING_M}m")
        print(f"  - Interpolation radius: {LIDAR_INTERPOLATION_RADIUS_M}m")
    
    # Check for required libraries
    if not PYPROJ_AVAILABLE:
        print("\n" + "!"*70)
        print("WARNING: pyproj is not installed!")
        print("This is REQUIRED for LiDAR coordinate transformation.")
        print("Install with: pip install pyproj")
        print("!"*70 + "\n")
    
    print("="*70)
    
    try:
        # Initialize cache and scan existing elevation files
        print("\nInitializing smart cache system...")
        cache_init = SmartInterpolationCache(CACHE_DIR)
        
        # Initialize spatial elevation cache
        print("\nInitializing spatial elevation cache...")
        spatial_cache = SpatialElevationCache(CACHE_DIR)
        
        # Scan existing elevation files to build global cache
        cache_init.scan_existing_elevation_files(OUTPUT_DIR)
        
        # Extract and interpolate route
        print("\nSTEP 1: Processing route...")
        coordinates = extract_coordinates_from_kmz_with_path_fixing(INPUT_KMZ)  # Note: changed function name
        interpolated_points = interpolate_route_with_smart_cache(
            coordinates, 
            INTERPOLATION_SPACING_M,
            CACHE_DIR,
            INPUT_KMZ
        )
        
        # Extract manual elevation waypoints
        elevation_waypoints = extract_manual_elevation_waypoints(INPUT_KMZ)
        
        # Apply manual elevation waypoints if any
        if elevation_waypoints:
            interpolated_points = apply_manual_elevation_waypoints(
                interpolated_points, elevation_waypoints
            )
        
        # Check spatial cache for elevations
        print("\nSTEP 1.5: Checking spatial elevation cache...")
        original_missing = sum(1 for p in interpolated_points if p.elevation is None)
        
        if OVERRIDE_CACHE:
            print("  Override mode active - will clear LiDAR points from cache")
            
            # First, use spatial cache normally to get all cached elevations
            interpolated_points, initial_cache_hits = spatial_cache.process_points(interpolated_points)
            
            # Now clear elevations that came from LiDAR sources
            lidar_cleared = 0
            for p in interpolated_points:
                # Clear elevation if it came from LiDAR
                if p.elevation is not None and p.data_source == 'lidar':
                    p.elevation = None
                    p.data_source = None
                    p.source_priority = None
                    lidar_cleared += 1
            
            # Recalculate statistics
            cache_hits = initial_cache_hits - lidar_cleared
            current_missing = sum(1 for p in interpolated_points if p.elevation is None)
            
            print(f"  Initially found in cache: {initial_cache_hits:,}")
            print(f"  LiDAR points cleared for re-extraction: {lidar_cleared:,}")
            print(f"  Net cache hits (non-LiDAR): {cache_hits:,}")
        else:
            # Normal spatial cache check
            interpolated_points, cache_hits = spatial_cache.process_points(interpolated_points)
            current_missing = sum(1 for p in interpolated_points if p.elevation is None)
        
        print(f"  Points needing elevation: {original_missing:,}")
        print(f"  Found in spatial cache: {cache_hits:,}")
        print(f"  Still need extraction: {current_missing:,}")
        
        # Skip tile search if all points found in cache (unless override is active)
        if current_missing == 0 and not OVERRIDE_CACHE:
            print("\n✓ ALL ELEVATIONS FOUND IN SPATIAL CACHE!")
            print("  Skipping tile search, download, and extraction...")
            tiles_by_type = {}
            bridge_tunnel_segments = []
            
            # Still check for bridge/tunnel segments if provided
            if BRIDGE_TUNNEL_KMZ:
                print("\nSTEP 1.5: Extracting bridge/tunnel segments...")
                bridge_tunnel_segments = extract_bridge_tunnel_segments(BRIDGE_TUNNEL_KMZ)
        elif current_missing == 0 and OVERRIDE_CACHE:
            # This shouldn't happen with the clearing above, but handle it anyway
            print("\n⚠️ Override mode but no LiDAR points found to re-extract")
            print("  Continuing to search for LiDAR tiles anyway...")
            # Don't skip the tile search
        else:
            # Normal flow continues here (tile search will happen)
            # Step 1.5: Extract bridge/tunnel segments if provided
            bridge_tunnel_segments = []
            if BRIDGE_TUNNEL_KMZ:
                print("\nSTEP 1.5: Extracting bridge/tunnel segments...")
                bridge_tunnel_segments = extract_bridge_tunnel_segments(BRIDGE_TUNNEL_KMZ)
            
            # Step 2: Search for elevation data tiles
            print("\nSTEP 2: Searching for elevation data tiles...")
            searcher = PointBasedElevationSearcher()
            # Only search for points without elevations
            points_needing_extraction = [p for p in interpolated_points if p.elevation is None]
            # Pass both the filtered list AND the full original list
            tiles_by_type = searcher.search_tiles_for_points(points_needing_extraction, interpolated_points)
            
            print("\nSTEP 3: Downloading elevation data...")
            downloader = ElevationDataDownloader()  # ← ADD THIS LINE
            
            for data_type, tiles in tiles_by_type.items():
                if data_type != 'opentopo':
                    print(f"\nDownloading {data_type} tiles...")
                    downloaded = downloader.download_tiles_parallel(tiles)
                    print(f"  Downloaded {len(downloaded)}/{len(tiles)} tiles")
            
            # Step 4: Extract elevation values
            print("\nSTEP 4: Extracting elevation values...")
            extractor = ElevationExtractor()
            cached_count = sum(1 for p in interpolated_points if p.elevation is not None)
            if cached_count > 0:
                print(f"  Skipping extraction for {cached_count} points with cached elevations")
                
                # Filter tiles to only process uncached points
                for data_type, tiles in tiles_by_type.items():
                    for tile in tiles:
                        # Update covered_points to exclude already cached points
                        original_count = len(tile.covered_points)
                        tile.covered_points = {
                            idx for idx in tile.covered_points 
                            if interpolated_points[idx].elevation is None
                        }
                        filtered_count = len(tile.covered_points)
                        if filtered_count < original_count:
                            print(f"    Filtered {data_type} tile {tile.name}: {original_count} -> {filtered_count} points")
            interpolated_points = extractor.extract_elevations(tiles_by_type, interpolated_points)
    
            # Merge override cache if enabled  
            if OVERRIDE_CACHE:
                extractor.merge_override_cache()
            
            # USE THIS VERSION to update spatial cache with corrected data:
            if OVERRIDE_CACHE:
                print("\nUpdating spatial cache with CORRECTED elevations...")
                # The spatial cache will now have the fixed LiDAR elevations
                spatial_cache.add_extracted_elevations(interpolated_points)
                print("  ✓ Spatial cache updated with corrected LiDAR elevations")
            else:
                print("\nUpdating spatial cache with extracted elevations...")
                spatial_cache.add_extracted_elevations(interpolated_points)
        
        # Step 4.5: Process elevation data
        print("\nSTEP 4.5: Processing elevation data...")
        processor = ElevationProcessor()
        
        # Apply manual bridge/tunnel leveling FIRST if segments provided
        if bridge_tunnel_segments:
            interpolated_points = processor.apply_bridge_tunnel_leveling(
                interpolated_points, bridge_tunnel_segments
            )
        
        # Then filter remaining bridges, tunnels, and outliers
        interpolated_points = processor.filter_bridges_and_tunnels(interpolated_points)
        
        # Finally apply smoothing
        interpolated_points = processor.smooth_elevations(interpolated_points)
        
        # Step 5: Save results
        print("\nSTEP 5: Saving results...")
        output_file = OUTPUT_DIR / f"{Path(INPUT_KMZ).stem}_elevation_data.csv"
        if OVERRIDE_CACHE and 'extractor' in locals():
            extractor.merge_override_cache()
        
        # UPDATE SMART CACHE WITH EXTRACTED ELEVATIONS
        print("\nUpdating smart cache with extracted elevations...")
        cache_for_update = SmartInterpolationCache(CACHE_DIR)
        
        # Get the KMZ hash
        kmz_hash = cache_for_update.get_kmz_hash(INPUT_KMZ)
        
        # Update the cache with final elevations
        cache_for_update._update_cache(interpolated_points, kmz_hash)
        
        df = pd.DataFrame([
            {
                'Index': p.index,
                'Latitude': p.lat,
                'Longitude': p.lon,
                'Distance_m': p.distance_m,
                'Elevation_ft': p.elevation if p.elevation is not None else np.nan,  # Now in feet
                'DataSource': p.data_source,
                'Priority': p.source_priority,
                'IsInterpolated': p.is_interpolated
            }
            for p in interpolated_points
        ])
        
        df.to_csv(output_file, index=False)
        print(f"  Saved elevation data to: {output_file}")
        
        # Summary
        print("\n" + "="*70)
        print("SUMMARY")
        print("="*70)
        print(f"Route distance: {interpolated_points[-1].distance_m/1000:.2f} km")
        print(f"Total points: {len(interpolated_points)}")
        
        total_tiles = sum(len(tiles) for tiles in tiles_by_type.values())
        print(f"\nTiles found: {total_tiles}")
        
        sources_searched = set(tiles_by_type.keys())
        all_sources = ['lidar', 'dem_sub1m', 'dem_1m', 'dem_10m', 'opentopo']
        
        print("\nData source search status:")
        for src in all_sources:
            if src in sources_searched:
                tiles = tiles_by_type[src]
                count = sum(1 for p in interpolated_points if p.data_source == src)
                print(f"  ✓ {src}: SEARCHED - {len(tiles)} tiles, {count} points")
                
                if src == 'lidar' and LIDAR_INTERPOLATION_ENABLED:
                    direct_count = sum(1 for p in interpolated_points 
                                     if p.data_source == 'lidar' and not p.is_interpolated)
                    interp_count = sum(1 for p in interpolated_points 
                                     if p.data_source == 'lidar' and p.is_interpolated)
                    if interp_count > 0:
                        print(f"      Direct: {direct_count} points")
                        print(f"      Interpolated: {interp_count} points")
            elif src in ['opentopo']:
                print(f"  ○ {src}: SKIPPED (not needed)")
            else:
                print(f"  ✗ {src}: NOT NEEDED (better sources available)")
        
        # Coverage analysis
        points_with_elevation = sum(1 for p in interpolated_points if p.elevation is not None)
        coverage_percent = (points_with_elevation / len(interpolated_points)) * 100 if interpolated_points else 0
        
        print(f"\nElevation coverage: {points_with_elevation}/{len(interpolated_points)} ({coverage_percent:.1f}%)")
        
        if points_with_elevation < len(interpolated_points):
            missing_points = len(interpolated_points) - points_with_elevation
            print(f"  WARNING: {missing_points} points without elevation data")
        
        # Data source distribution
        source_counts = {}
        for p in interpolated_points:
            if p.data_source:
                source_counts[p.data_source] = source_counts.get(p.data_source, 0) + 1
        
        if source_counts:
            print("\nElevation sources used:")
            for source, count in sorted(source_counts.items(), key=lambda x: x[1], reverse=True):
                percent = (count / len(interpolated_points)) * 100
                print(f"  {source}: {count} points ({percent:.1f}%)")
        
        # Bridge/tunnel summary
        if BRIDGE_TUNNEL_FILTER_ENABLED:
            bridge_count = sum(1 for p in interpolated_points 
                             if hasattr(p, 'original_elevation_bridge'))
            tunnel_count = sum(1 for p in interpolated_points 
                             if hasattr(p, 'original_elevation_tunnel'))
            
            if bridge_count > 0 or tunnel_count > 0:
                print(f"\nBridge/Tunnel filtering:")
                if bridge_count > 0:
                    print(f"  Bridges detected and leveled: {bridge_count} points")
                if tunnel_count > 0:
                    print(f"  Tunnels detected and leveled: {tunnel_count} points")
        
        # Smoothing summary
        if SMOOTHING_ENABLED:
            print(f"\nSmoothing applied: {SMOOTHING_METHOD}")
            print(f"  Window size: {SMOOTHING_WINDOW} points")
        
        # Print spatial cache statistics
        spatial_cache.print_session_stats()
        
        print("\n" + "="*70)
        print("PROCESSING COMPLETE")
        print("="*70)
        
    except Exception as e:
        print(f"\nError in main pipeline: {e}")
        import traceback
        traceback.print_exc()
        raise

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Elevation Profile Visualizer and 3D KMZ Exporter
Creates visualizations with corrected gradient calculations
Excludes manual elevation waypoints from export
"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import zipfile
import os
import xml.etree.ElementTree as ET
import tempfile
from typing import List, Tuple, Optional, Dict
import simplekml
from datetime import datetime
import re
import warnings
warnings.filterwarnings('ignore')

# ==================== CONFIGURATION ====================

# Input files
INPUT_CSV = r"C:\Users\peckb\Documents\TRIMET\CVG-Express_elevation_data.csv"
INPUT_KMZ = r"C:\Users\peckb\Documents\TRIMET\CVG-Express.kmz"

# Output configuration
OUTPUT_DIR = Path(INPUT_CSV).parent
VISUALIZATION_DPI = 150
PROFILE_FIGURE_SIZE = (16, 10)
VERTICAL_EXAGGERATION = 10.0
GRADE_THRESHOLD = 4.0  # Percent grade for highlighting

# KMZ settings
KMZ_LINE_WIDTH = 4
KMZ_ALTITUDE_MODE = 'absolute'
KMZ_EXTRUDE_TO_GROUND = True
KMZ_COLOR_BY_SOURCE = True

# Unit conversion
FEET_TO_METERS = 0.3048

# ==================== DATA LOADING ====================

def load_elevation_data(csv_path: str) -> pd.DataFrame:
    """Load elevation data from CSV file"""
    print(f"Loading elevation data from {os.path.basename(csv_path)}...")
    
    df = pd.read_csv(csv_path)
    
    # Handle elevation column
    if 'Elevation_ft' in df.columns:
        df['Elevation'] = df['Elevation_ft']
        print("  Found elevation data in feet")
    elif 'Elevation' not in df.columns:
        raise ValueError("No elevation column found in CSV")
    
    # Convert to meters for KMZ
    df['Elevation_m'] = df['Elevation'] * FEET_TO_METERS
    
    # Handle NaN values
    nan_count = df['Elevation'].isna().sum()
    if nan_count > 0:
        print(f"  Warning: {nan_count} points have missing elevation data")
        df['Elevation'].interpolate(method='linear', inplace=True)
        df['Elevation_m'].interpolate(method='linear', inplace=True)
    
    print(f"  Loaded {len(df)} points")
    
    # Print data source distribution
    if 'DataSource' in df.columns:
        source_counts = df['DataSource'].value_counts().to_dict()
        print(f"  Data sources: {source_counts}")
    
    return df

def extract_kmz_waypoints(kmz_path: str, exclude_elevation_waypoints: bool = True) -> List[Dict]:
    """Extract waypoint Placemarks from KMZ file, optionally excluding manual elevation waypoints"""
    print(f"Extracting waypoints from {os.path.basename(kmz_path)}...")
    
    waypoints = []
    
    # Pattern to match "ELEVATION = XXX FT" (case insensitive)
    elevation_pattern = re.compile(r'ELEVATION\s*=\s*\d+(?:\.\d+)?\s*FT', re.IGNORECASE)
    
    with tempfile.TemporaryDirectory() as temp_dir:
        with zipfile.ZipFile(kmz_path, 'r') as kmz:
            kmz.extractall(temp_dir)
            
            kml_file = None
            for file in os.listdir(temp_dir):
                if file.endswith('.kml'):
                    kml_file = os.path.join(temp_dir, file)
                    break
            
            if not kml_file:
                raise ValueError("No KML file found in KMZ")
            
            tree = ET.parse(kml_file)
            root = tree.getroot()
    
    # Check for namespace
    ns = {'kml': 'http://www.opengis.net/kml/2.2'}
    if not root.findall('.//kml:Point', ns):
        ns = {}
    
    # Extract ONLY Point placemarks (actual waypoints), NOT LineString vertices
    if ns:
        placemarks = root.findall('.//kml:Placemark', ns)
    else:
        placemarks = root.findall('.//Placemark')
    
    excluded_count = 0
    
    for placemark in placemarks:
        # Only process Placemarks with Points (waypoints), skip LineStrings
        if ns:
            point = placemark.find('.//kml:Point', ns)
            name_elem = placemark.find('kml:name', ns)
        else:
            point = placemark.find('.//Point')
            name_elem = placemark.find('name')
        
        if point is not None:
            # This is an actual waypoint/placemark
            waypoint_name = name_elem.text if name_elem is not None else "Waypoint"
            
            # Check if this is a manual elevation waypoint and skip if requested
            if exclude_elevation_waypoints and elevation_pattern.search(waypoint_name):
                excluded_count += 1
                continue
            
            if ns:
                coords_elem = point.find('kml:coordinates', ns)
            else:
                coords_elem = point.find('coordinates')
            
            if coords_elem is not None:
                coords_text = coords_elem.text.strip()
                parts = coords_text.split(',')
                if len(parts) >= 2:
                    waypoints.append({
                        'name': waypoint_name,
                        'latitude': float(parts[1]),
                        'longitude': float(parts[0]),
                        'is_start': False,  # Will be determined by position
                        'is_end': False
                    })
    
    # If no Point placemarks found, return empty list (no waypoints to add)
    if not waypoints and excluded_count == 0:
        print("  No waypoint Placemarks found in KMZ (only LineStrings)")
        return []
    
    if excluded_count > 0:
        print(f"  Excluded {excluded_count} manual elevation waypoints")
    
    print(f"  Found {len(waypoints)} waypoint Placemarks to include")
    return waypoints

def parse_elevation_from_name(name: str) -> Optional[float]:
    """Extract elevation value from waypoint name if it contains elevation data"""
    import re
    # Pattern to match elevation in feet (e.g., "125 FT", "125.5 ft", etc.)
    elevation_pattern = re.compile(r'(\d+(?:\.\d+)?)\s*FT', re.IGNORECASE)
    match = elevation_pattern.search(name)
    if match:
        return float(match.group(1))
    return None

# ==================== GRADIENT CALCULATION ====================

def calculate_grades(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate corrected gradients between points with better filtering"""
    df = df.copy()
    
    # Calculate differences
    df['Distance_Diff'] = df['Distance_m'].diff()
    df['Elevation_Diff'] = df['Elevation'].diff()
    
    # Calculate grade percentage (rise/run * 100)
    df['Grade_Percent'] = 0.0
    
    # Only calculate grade where we have meaningful distance
    # Ignore very small distances that can create artificial spikes
    MIN_DISTANCE_M = 1.0  # Minimum 1 meter to avoid division issues
    
    mask = df['Distance_Diff'] > MIN_DISTANCE_M
    
    # Calculate gradient in consistent units (meters)
    # Grade % = (rise in meters / run in meters) * 100
    elevation_diff_m = df.loc[mask, 'Elevation_Diff'] / 3.28084  # Convert feet to meters
    distance_diff_m = df.loc[mask, 'Distance_Diff']
    
    df.loc[mask, 'Grade_Percent'] = (elevation_diff_m / distance_diff_m) * 100
    
    # Remove outliers before smoothing
    # Railway grades rarely exceed 4%, anything above 6% is likely noise
    OUTLIER_THRESHOLD = 6.0
    outlier_mask = df['Grade_Percent'].abs() > OUTLIER_THRESHOLD
    
    # For outliers, use interpolation from surrounding valid points
    if outlier_mask.any():
        valid_grades = df.loc[~outlier_mask, 'Grade_Percent']
        valid_indices = df.index[~outlier_mask]
        
        # Interpolate outliers
        for idx in df.index[outlier_mask]:
            # Find nearest valid points
            before_idx = valid_indices[valid_indices < idx]
            after_idx = valid_indices[valid_indices > idx]
            
            if len(before_idx) > 0 and len(after_idx) > 0:
                # Interpolate between nearest valid points
                before = before_idx[-1]
                after = after_idx[0]
                weight = (idx - before) / (after - before)
                df.loc[idx, 'Grade_Percent'] = (
                    (1 - weight) * df.loc[before, 'Grade_Percent'] + 
                    weight * df.loc[after, 'Grade_Percent']
                )
            elif len(before_idx) > 0:
                # Use last valid value
                df.loc[idx, 'Grade_Percent'] = df.loc[before_idx[-1], 'Grade_Percent']
            elif len(after_idx) > 0:
                # Use next valid value
                df.loc[idx, 'Grade_Percent'] = df.loc[after_idx[0], 'Grade_Percent']
    
    # Apply smoothing with larger window for railway (railways change grade gradually)
    # Use a window of about 100m (10 points at 10m spacing)
    SMOOTH_WINDOW = 11  # Should be odd
    df['Grade_Smooth'] = df['Grade_Percent'].rolling(
        window=SMOOTH_WINDOW, 
        center=True, 
        min_periods=3
    ).median()  # Use median to be more robust to remaining outliers
    
    # Fill any NaN values from smoothing
    df['Grade_Smooth'] = df['Grade_Smooth'].fillna(df['Grade_Percent'])
    
    # Final cap on extreme values (railway practical limits)
    df['Grade_Smooth'] = df['Grade_Smooth'].clip(-5, 5)
    
    return df

def calculate_statistics(df: pd.DataFrame) -> Dict:
    """Calculate route statistics using smoothed gradients"""
    stats = {
        'total_distance_km': df['Distance_m'].iloc[-1] / 1000,
        'min_elevation_ft': df['Elevation'].min(),
        'max_elevation_ft': df['Elevation'].max(),
        'total_ascent_ft': df[df['Elevation_Diff'] > 0]['Elevation_Diff'].sum(),
        'total_descent_ft': abs(df[df['Elevation_Diff'] < 0]['Elevation_Diff'].sum()),
        'avg_elevation_ft': df['Elevation'].mean(),
        'starting_elevation_ft': df['Elevation'].iloc[0],
        'ending_elevation_ft': df['Elevation'].iloc[-1],
        'net_elevation_change_ft': df['Elevation'].iloc[-1] - df['Elevation'].iloc[0]
    }
    
    # Grade statistics - use SMOOTHED grades, not raw
    if 'Grade_Smooth' in df.columns:
        valid_grades = df['Grade_Smooth'].dropna()
    else:
        # Fallback to Grade_Percent if smooth not available
        valid_grades = df['Grade_Percent'].dropna()
    
    if len(valid_grades) > 0:
        stats['max_grade_up'] = valid_grades.max()
        stats['max_grade_down'] = valid_grades.min()
        stats['avg_abs_grade'] = valid_grades.abs().mean()
    else:
        stats['max_grade_up'] = 0
        stats['max_grade_down'] = 0
        stats['avg_abs_grade'] = 0
    
    # Steep sections based on railway threshold (4%)
    RAIL_STEEP_THRESHOLD = 4.0
    stats['steep_up_percent'] = (valid_grades > RAIL_STEEP_THRESHOLD).sum() / len(df) * 100
    stats['steep_down_percent'] = (valid_grades < -RAIL_STEEP_THRESHOLD).sum() / len(df) * 100
    
    return stats

# ==================== VISUALIZATION ====================

def create_elevation_profile(df: pd.DataFrame, stats: Dict) -> plt.Figure:
    """Create elevation profile visualization"""
    print("\nCreating elevation profile...")
    
    fig = plt.figure(figsize=PROFILE_FIGURE_SIZE)
    
    # Convert distance to km
    distance_km = df['Distance_m'] / 1000
    
    # Elevation profile subplot
    ax1 = plt.subplot(3, 1, 1)
    
    # Plot elevation with data source coloring
    source_colors = {
        'lidar': 'purple',
        'dem_sub1m': 'blue',
        'dem_1m': 'green',
        'dem_10m': 'gold',
        'opentopo': 'orange',
        'manual': 'red',
        'smoothed_manual': 'pink',
        'none': 'gray'
    }
    
    if 'DataSource' in df.columns:
        for source, color in source_colors.items():
            mask = df['DataSource'] == source
            if mask.any():
                ax1.fill_between(
                    distance_km[mask],
                    df['Elevation'][mask].min() - 10,
                    df['Elevation'][mask],
                    alpha=0.3,
                    color=color,
                    label=source.upper()
                )
    
    # Main elevation line
    ax1.plot(distance_km, df['Elevation'], 'k-', linewidth=2, label='Elevation')
    
    # Mark start and end
    ax1.scatter(distance_km.iloc[0], df['Elevation'].iloc[0], 
                color='green', s=100, marker='o', zorder=5, label='Start')
    ax1.scatter(distance_km.iloc[-1], df['Elevation'].iloc[-1], 
                color='red', s=100, marker='s', zorder=5, label='End')
    
    # Stats text
    stats_text = (
        f"Distance: {stats['total_distance_km']:.2f} km\n"
        f"Start: {stats['starting_elevation_ft']:.1f} ft\n"
        f"End: {stats['ending_elevation_ft']:.1f} ft\n"
        f"Net: {stats['net_elevation_change_ft']:+.1f} ft\n"
        f"Range: {stats['min_elevation_ft']:.1f} - {stats['max_elevation_ft']:.1f} ft\n"
        f"Ascent: {stats['total_ascent_ft']:.1f} ft\n"
        f"Descent: {stats['total_descent_ft']:.1f} ft"
    )
    ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes,
            fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax1.set_ylabel('Elevation (feet)')
    ax1.set_title('Elevation Profile', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right', ncol=3, fontsize=9)
    
    # Grade profile subplot
    ax2 = plt.subplot(3, 1, 2, sharex=ax1)
    
    # Plot smoothed grade (use Grade_Smooth, not Grade_Percent)
    ax2.fill_between(distance_km, 0, df['Grade_Smooth'],
                     where=(df['Grade_Smooth'] >= 0),
                     color='green', alpha=0.5, label='Uphill')
    ax2.fill_between(distance_km, 0, df['Grade_Smooth'],
                     where=(df['Grade_Smooth'] < 0),
                     color='red', alpha=0.5, label='Downhill')
    
    # Railway grade threshold lines (4% is steep for rail)
    RAIL_GRADE_THRESHOLD = 4.0
    ax2.axhline(y=RAIL_GRADE_THRESHOLD, color='darkgreen', linestyle='--', alpha=0.5)
    ax2.axhline(y=-RAIL_GRADE_THRESHOLD, color='darkred', linestyle='--', alpha=0.5)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    
    # Calculate actual max values from smoothed data
    actual_max_up = df['Grade_Smooth'].max()
    actual_max_down = df['Grade_Smooth'].min()
    actual_avg_abs = df['Grade_Smooth'].abs().mean()
    
    # Grade stats (using smoothed values)
    grade_text = (
        f"Max Up: {actual_max_up:.1f}%\n"
        f"Max Down: {actual_max_down:.1f}%\n"
        f"Avg |Grade|: {actual_avg_abs:.2f}%"
    )
    ax2.text(0.98, 0.98, grade_text, transform=ax2.transAxes,
            fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax2.set_ylabel('Grade (%)')
    ax2.set_title('Gradient Analysis', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper left')
    
    # Set y-limits based on actual data with some padding
    y_max = max(abs(actual_max_up), abs(actual_max_down), RAIL_GRADE_THRESHOLD) * 1.2
    ax2.set_ylim(-y_max, y_max)
    
    # Data source coverage subplot
    ax3 = plt.subplot(3, 1, 3, sharex=ax1)
    
    if 'DataSource' in df.columns:
        y_pos = 0
        for source, color in source_colors.items():
            mask = df['DataSource'] == source
            if mask.any():
                coverage = mask.astype(float)
                ax3.fill_between(distance_km, y_pos, y_pos + coverage,
                               color=color, alpha=0.7, label=source.upper())
                y_pos += 1
        
        ax3.set_ylim(0, y_pos)
        ax3.set_yticks([])
        ax3.legend(loc='upper right', ncol=3, fontsize=9)
    
    ax3.set_xlabel('Distance (km)')
    ax3.set_ylabel('Data Source')
    ax3.set_title('Data Source Coverage', fontsize=12)
    ax3.grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Route Elevation Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    return fig

# ==================== KMZ EXPORT ====================

def match_waypoints_to_data(waypoints: List[Dict], df: pd.DataFrame) -> List[Dict]:
    """Match KMZ waypoints to elevation data points"""
    matched = []
    
    for wp in waypoints:
        # Find closest point in elevation data
        distances = np.sqrt(
            (df['Latitude'] - wp['latitude'])**2 + 
            (df['Longitude'] - wp['longitude'])**2
        )
        
        closest_idx = distances.idxmin()
        closest_row = df.loc[closest_idx]
        
        matched.append({
            'name': wp['name'],
            'latitude': wp['latitude'],
            'longitude': wp['longitude'],
            'elevation_ft': closest_row['Elevation'],
            'elevation_m': closest_row['Elevation_m'],
            'distance_km': closest_row['Distance_m'] / 1000,
            'data_source': closest_row.get('DataSource', 'N/A'),
            'grade': closest_row.get('Grade_Percent', 0),
            'is_start': wp['is_start'],
            'is_end': wp['is_end']
        })
    
    return matched

def create_kmz_export(df: pd.DataFrame, waypoints: List[Dict], output_path: str):
    """Create KMZ file with elevation path and filtered waypoints (no manual elevation waypoints)"""
    print(f"\nCreating KMZ export...")
    
    kml = simplekml.Kml(name=f"Elevation Profile - {Path(output_path).stem}")
    
    # Main folder
    main_folder = kml.newfolder(name="Route with Elevation")
    
    # Create elevation path (the 3D line with elevation)
    path_folder = main_folder.newfolder(name="Elevation Path")
    
    # Color mapping for data sources
    source_colors = {
        'lidar': simplekml.Color.purple,
        'dem_sub1m': simplekml.Color.blue,
        'dem_1m': simplekml.Color.green,
        'dem_10m': simplekml.Color.yellow,
        'opentopo': simplekml.Color.orange,
        'manual': simplekml.Color.red,
        'smoothed_manual': simplekml.Color.lightpink,
        'none': simplekml.Color.gray
    }
    
    if KMZ_COLOR_BY_SOURCE and 'DataSource' in df.columns:
        # Create paths by data source
        for source in df['DataSource'].unique():
            if pd.isna(source):
                continue
            
            source_df = df[df['DataSource'] == source]
            if len(source_df) < 2:
                continue
            
            coords = [(row['Longitude'], row['Latitude'], row['Elevation_m']) 
                     for _, row in source_df.iterrows()]
            
            path = path_folder.newlinestring(
                name=f"{source.upper()} Coverage",
                coords=coords
            )
            path.style.linestyle.color = source_colors.get(source, simplekml.Color.gray)
            path.style.linestyle.width = KMZ_LINE_WIDTH
            path.altitudemode = simplekml.AltitudeMode.absolute
            path.extrude = 1 if KMZ_EXTRUDE_TO_GROUND else 0
    else:
        # Single path
        coords = [(row['Longitude'], row['Latitude'], row['Elevation_m']) 
                 for _, row in df.iterrows()]
        
        path = path_folder.newlinestring(
            name="Elevation Path",
            coords=coords
        )
        path.style.linestyle.color = simplekml.Color.red
        path.style.linestyle.width = KMZ_LINE_WIDTH
        path.altitudemode = simplekml.AltitudeMode.absolute
        path.extrude = 1 if KMZ_EXTRUDE_TO_GROUND else 0
    
    # Add filtered waypoints (no manual elevation waypoints)
    if waypoints:
        waypoint_folder = main_folder.newfolder(name="Waypoints")
        
        for wp in waypoints:
            # Determine icon color
            if wp['is_start']:
                icon_color = simplekml.Color.green
                icon_scale = 1.0
            elif wp['is_end']:
                icon_color = simplekml.Color.red
                icon_scale = 1.0
            else:
                icon_color = simplekml.Color.yellow
                icon_scale = 0.7
            
            pnt = waypoint_folder.newpoint(
                name=wp['name'],
                coords=[(wp['longitude'], wp['latitude'], wp['elevation_m'])]
            )
            
            pnt.description = (
                f"Distance: {wp['distance_km']:.2f} km\n"
                f"Elevation: {wp['elevation_ft']:.1f} ft ({wp['elevation_m']:.1f} m)\n"
                f"Grade: {wp['grade']:.1f}%\n"
                f"Data Source: {wp['data_source']}"
            )
            
            pnt.style.iconstyle.color = icon_color
            pnt.style.iconstyle.scale = icon_scale
            pnt.altitudemode = simplekml.AltitudeMode.absolute
    
    # Save KMZ
    kml.savekmz(output_path)
    print(f"  Saved to: {output_path}")

# ==================== MAIN ====================

def main():
    """Main pipeline"""
    print("="*70)
    print("ELEVATION PROFILE VISUALIZER")
    print("="*70)
    
    try:
        # Load data
        print("\nStep 1: Loading data...")
        df = load_elevation_data(INPUT_CSV)
        
        # Calculate grades
        print("\nStep 2: Calculating gradients...")
        df = calculate_grades(df)
        
        # Calculate statistics
        stats = calculate_statistics(df)
        
        print(f"\nStatistics:")
        print(f"  Distance: {stats['total_distance_km']:.2f} km")
        print(f"  Elevation: {stats['starting_elevation_ft']:.1f} → {stats['ending_elevation_ft']:.1f} ft")
        print(f"  Net change: {stats['net_elevation_change_ft']:+.1f} ft")
        print(f"  Max grade: +{stats['max_grade_up']:.1f}% / {stats['max_grade_down']:.1f}%")
        
        # Create visualization
        print("\nStep 3: Creating visualization...")
        fig = create_elevation_profile(df, stats)
        
        # Save visualization
        output_file = OUTPUT_DIR / f"{Path(INPUT_CSV).stem}_profile.png"
        fig.savefig(output_file, dpi=VISUALIZATION_DPI, bbox_inches='tight')
        print(f"  Saved profile to: {output_file}")
        
        # Extract and process KMZ waypoints (excluding manual elevation waypoints)
        print("\nStep 4: Processing KMZ waypoints...")
        waypoints = extract_kmz_waypoints(INPUT_KMZ, exclude_elevation_waypoints=True)
        
        if waypoints:
            matched_waypoints = match_waypoints_to_data(waypoints, df)
            print(f"  Matched {len(matched_waypoints)} waypoints to elevation data")
        else:
            matched_waypoints = []
            print("  No regular waypoints found (only elevation waypoints or LineStrings)")
        
        # Create KMZ export
        print("\nStep 5: Creating KMZ export...")
        kmz_output = OUTPUT_DIR / f"{Path(INPUT_CSV).stem}_3d.kmz"
        create_kmz_export(df, matched_waypoints, str(kmz_output))
        
        print("\n" + "="*70)
        print("COMPLETE")
        print("="*70)
        print(f"✓ Elevation profile: {output_file.name}")
        print(f"✓ KMZ with elevation: {kmz_output.name}")
        if matched_waypoints:
            print(f"  - {len(matched_waypoints)} waypoints included")
        else:
            print(f"  - No regular waypoints (manual elevation waypoints excluded)")
        
        # Show plot
        plt.show()
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()